# Order and density of a virtual swarm modulate directional alignment in marching locusts

Juvenile desert locusts (*Schistocerca gregaria*) align their heading with the
direction of neighbor motion, a core ingredient of marching behavior. In
virtual-reality experiments (Sayin et al., 2025), focal locusts walked on a
motion-compensating spherical treadmill surrounded by a panoramic projection
of moving **virtual conspecifics**. The virtual swarm was defined by two
continuous stimulus parameters:

- **order** — the coherence of the virtual motion, 0 = disordered to 1 =
  perfectly aligned (six levels: 0, 0.2, 0.4, 0.6, 0.8, 1.0);
- **density** — the number of virtual locusts per m$^2$ (six levels: 1, 4, 8,
  16, 32, 64).

The biological question: is the focal locust's directional alignment driven
primarily by the directional structure of neighbor motion (order), or does the
number of neighbors (density) matter comparably?

This notebook re-analyzes one such dataset with **von Mises GLMs** on
re-discretized step headings: it builds the step-level design, screens
candidate fixed-effect specifications with PSIS-LOO at the level of
individuals, fits the final random-intercept hierarchical model in `bambi`
with `unwrap`'s circular link functions, and validates it with the
**alignment** predictive statistic,

$$\mathcal{A}(\mathbf{x}) \;=\; R(\kappa(\mathbf{x}))\,\cos\bigl(\mu(\mathbf{x})\bigr),
\qquad R(\kappa)=I_1(\kappa)/I_0(\kappa),$$

measured at the virtual march direction ($0^\circ$ — perfect alignment is
$\mathcal{A}=1$; no preferred direction is $\mathcal{A}=0$).

> **Note on data availability:** the raw per-animal tracking parts
(``density_order/XY_part_*.parquet``, ~870 MB in
[jgraving/unwrap-data](https://github.com/jgraving/unwrap-data)) are not
shipped with this repository and are downloaded on first use by
`load_density_order()` (first call shows a progress meter; later runs
reuse the cached clone with no download). The rediscretization below
runs in-notebook, so the notebook is fully self-contained after that
initial fetch.

In [ ]:
from pathlib import Path
from unwrap.datasets import load_density_order

# `data_root` may be:
#   None (default) -> pulls/clones ``unwrap-data`` to ``~/.cache/unwrap-data``
#   a local path to an existing ``unwrap-data`` checkout -> skips the network
#   a path to a directory that already contains the parquet parts
# `columns_only` (optional) limits which columns get read.
dfXY = load_density_order()
# ---------------------------------------------------------------------------
# Figures are drawn by a sibling plotting script (locust_density_plots.py);
# the notebook keeps the statistical code. Locate the script whether this
# notebook runs from notebooks/ or from the repository root.
# ---------------------------------------------------------------------------
import sys
for _d in (Path.cwd(), Path.cwd() / "notebooks"):
    if (_d / "locust_density_plots.py").is_file():
        if str(_d) not in sys.path:
            sys.path.insert(0, str(_d))
        break
import locust_density_plots as plots

plots.set_style()


Figures will be saved to: figures/locust


## Setup

From the repository root, in a terminal:

```bash
uv venv --python 3.12
uv pip install -e ".[notebook]"
uv run --no-project jupyter lab notebooks/locust_density_order.ipynb
```

The `.[notebook]` extra covers everything this notebook imports.

## Data loading

The first code cell pulls the raw per-animal tracking parts from
[jgraving/unwrap-data](https://github.com/jgraving/unwrap-data) with
`load_density_order()` (one-time ~870 MB download with a progress bar; later
runs reuse the cached clone). The rediscretization code cell next samples each
raw trajectory to equal arc length — one step every **6 body lengths** — and
builds the step-level ``merged_df`` the models are fit to. Columns of
``merged_df``:

- `fname` — raw individual identifier (one raw trajectory = one animal);
- `order`, `density` — the virtual-swarm stimulus condition, constant per
  animal (each animal experienced exactly one of the
  $6 \times 6 = 36$ order–density combinations);
- `X`, `Y` — step position on the arena plane (meters);
- `dX`, `dY` — the step's displacement;
- `direction` — step heading $\theta = \operatorname{atan2}(dY, dX)$ in
  radians, with $0$ the global march direction;
- `animal_id` — zero-based id, unique per (order, density, `fname`).


In [ ]:
# def _sorted_unique(series):
#     vals = series.dropna().unique().tolist()
#     try:
#         # Try numeric sort first
#         return sorted(vals, key=float)
#     except Exception:
#         # Fall back to default sort for non-numeric types
#         return sorted(vals)

# orders = _sorted_unique(dfXY['order'])
# densities = _sorted_unique(dfXY['density'])

# print("Sorted orders:", orders)
# print("Sorted densities:", densities)

# # --- Make a grid of subplots: rows=order, cols=density ---
# n_rows = len(orders)
# n_cols = len(densities)

# plt.rcParams.update({
#     'font.size': 8,
#     'ytick.major.width': 2,
#     'xtick.major.width': 2,
#     'axes.linewidth': 2,
#     'font.family': 'Arial'
# })

# fig, axes = plt.subplots(
#     n_rows, n_cols,
#     figsize=(n_cols * 1.4, n_rows * 1.4),
#     dpi=300,
#     squeeze=False
# )

# # Custom colormap (same as before)
# start_color = '#30013b'  # Dark Purple
# end_color = '#f2cf26'    # Yellow
# cmap = plt.cm.tab20#LinearSegmentedColormap.from_list('custom_cmap', [start_color, end_color])

# for r, o in enumerate(orders):
#     for c, d in enumerate(densities):
#         ax = axes[r, c]
#         df_sub = dfXY.loc[(dfXY['order'] == o) & (dfXY['density'] == d)]

#         if df_sub.empty:
#             # No data for this combo—keep frame for consistency
#             ax.set_title(f"d={d} | o={o}", fontsize=8)
#             ax.set_xlim(-8, 8)
#             ax.set_ylim(-8, 8)
#             ax.set_xticks([-8, 0, 8])
#             ax.set_yticks([-8, 0, 8])
#             ax.set_aspect('equal')
#             ax.text(0.5, 0.5, "no data", ha='center', va='center', transform=ax.transAxes, alpha=0.6)
#             continue

#         grouped = df_sub.groupby('fname', sort=False)

#         # Plot each trajectory
#         total = len(grouped)
#         # Avoid division by zero; total can't be 0 here but guard anyway
#         denom = max(total, 1)
#         for i, (_, grp) in enumerate(grouped):
#             xx = grp["X"].values
#             yy = grp["Y"].values
#             ax.plot(xx, yy, color=cmap(i / denom), linewidth=0.8)

#         # Axes formatting
#         ax.set_xlim(-8, 8)
#         ax.set_ylim(-8, 8)
#         ax.set_xticks([-8, 0, 8])
#         ax.set_yticks([-8, 0, 8])
#         ax.set_aspect('equal')
#         ax.set_title(f"d={d} | o={o}", fontsize=8)

# # Optional: nicer outer labels
# fig.suptitle("Trajectories by Order (rows) × Density (cols)", fontsize=9, y=1.02)
# fig.tight_layout()

# # Show and/or save
# plt.show()


In [ ]:
import numpy as np
np.sort(dfXY["density"].unique()), np.sort(dfXY["order"].unique())

(array([ 1,  4,  8, 16, 32, 64]), array([0. , 0.2, 0.4, 0.6, 0.8, 1. ]))

In [ ]:
dfXY["fname"].unique().shape

(470,)

In [ ]:
dfXY["loss"].min(), dfXY["loss"].max()

(np.float64(0.0008416736806140523), np.float64(0.9999749997916649))

## Stimulus levels and animals

The observed stimulus levels match the experimental design —
$\mathrm{density}\in\{1,4,8,16,32,64\}\ \mathrm{m}^{-2}$ and
$\mathrm{order}\in\{0,0.2,0.4,0.6,0.8,1.0\}$ — and the file set covers the
raw individual tracks (470); after trajectory re-discretization, $N = 440$
animals enter modelling.


In [ ]:
dfXY

,X,Y,fname,loss,order,density,groups
0,-0.000020,-0.000168,20220829_1234,0.011458,0.4,16,0.4_16
1,-0.000041,-0.000345,20220829_1234,0.011458,0.4,16,0.4_16
2,-0.000062,-0.000527,20220829_1234,0.011458,0.4,16,0.4_16
3,-0.000084,-0.000719,20220829_1234,0.011458,0.4,16,0.4_16
4,-0.000106,-0.000912,20220829_1234,0.011458,0.4,16,0.4_16
...,...,...,...,...,...,...,...
53226539,3.789826,-0.082112,20221104_1401,0.016808,1.0,4,1.0_04
53226540,3.789826,-0.082110,20221104_1401,0.016808,1.0,4,1.0_04
53226541,3.789826,-0.082114,20221104_1401,0.016808,1.0,4,1.0_04
53226542,3.789826,-0.082117,20221104_1401,0.016808,1.0,4,1.0_04


In [ ]:
import locust_density_preprocess as dens

# Rediscretize every raw trial (X,Y) to 6 body lengths along-track -> merged_df
# (carries order, density, dX/dY, direction, and the dense numeric animal_id).
# Moved to notebooks/locust_density_preprocess.py; bit-identical to the old
# inline cell on the current unwrap-data (18,641 steps).
merged_df = dens.rediscretize_density_order(dfXY)

merged_df


## How the step data were produced

Raw arena positions were resampled at equal arc length: each trajectory was
densely interpolated, then sampled at successive points spaced by the
**step length of 6 body lengths** (chosen by a lag-1 circular-autocorrelation
screen over $\ell \in \{1,\dots,10\}$ body lengths, to obtain
approximately independent steps). Each consecutive pair of resampled
positions defines a step heading
$\theta_t = \operatorname{atan2}(dY_t, dX_t)$.

Some animals were dropped during rediscretization (too short, or with
tracking gaps), so 440 animals enter modelling. The rediscretization runs in the code cells of this notebook
(dense interpolation, then equal-arc-length stepping); it reproduces, for any
raw parts, the same steps the data clone publishes are derived from.


In [ ]:
merged_df

,fname,order,density,X,Y,dX,dY,direction,animal_id
0,20220829_1234,0.4,16,0.146888,-0.189951,0.146908,-0.189784,-0.912061,0
1,20220829_1234,0.4,16,-0.086330,-0.133167,-0.233218,0.056785,2.902756,0
2,20220829_1234,0.4,16,-0.310468,-0.219125,-0.224138,-0.085959,-2.775384,0
3,20220829_1234,0.4,16,-0.434566,-0.424557,-0.124098,-0.205432,-2.114212,0
4,20220829_1234,0.4,16,-0.623218,-0.276234,-0.188653,0.148323,2.475310,0
...,...,...,...,...,...,...,...,...,...
18636,20221104_1401,1.0,4,2.843251,-0.021066,0.239904,-0.005247,-0.021868,439
18637,20221104_1401,1.0,4,3.059277,0.083800,0.216027,0.104867,0.451928,439
18638,20221104_1401,1.0,4,3.295434,0.041053,0.236157,-0.042747,-0.179073,439
18639,20221104_1401,1.0,4,3.535442,0.037230,0.240008,-0.003824,-0.015930,439


In [ ]:
merged_df["animal_id"].nunique()

440

## Mean step vectors and the modeling table

- **Trajectory plot** (first figure): per-animal re-discretized paths for
  several order–density conditions — higher-order conditions show visibly
  rectilinear, march-aligned paths.
- **Mean-vector plot** (second figure): each animal's mean resultant step
  vector as a ray from the origin inside a unit circle; the ray's angle is the
  animal's mean heading and its length the per-animal mean resultant length.

The modeling table then prepares the heading response (wrapped to
$(-\pi, \pi]$) and the normalized predictors used in all models:
$\tilde{o} = 2\,\mathrm{order} - 1$ and
$\tilde{d} = 2\log_2(\mathrm{density})/\log_2(64) - 1$, both in
$[-1, 1]$, mean-centered at fit time. For **model selection**, each animal is
summarized by the mean resultant direction of its steps — one circular
observation per animal — so the five candidate fixed-effect specifications
(intercept only, order only, density only, additive, interaction) can be
compared with PSIS-LOO at the level of individuals: a fast, individual-level
stand-in for leave-one-individual-out cross-validation.


In [ ]:
# # ---------- OUTLIER REMOVAL AFTER REDISCRETIZATION ----------

# import numpy as np
# import pandas as pd

# # 1) Build per-trial features from merged_df
# #    - mean sin/cos of heading (circular mean embedding)
# feat_df = merged_df.copy()

# feat_df["step_len"] = np.hypot(feat_df["dX"], feat_df["dY"])
# feat_df["sin"] = np.sin(feat_df["direction"])
# feat_df["cos"] = np.cos(feat_df["direction"])

# trial_features = (
#     feat_df
#     .groupby(["order", "density", "fname"], sort=False)
#     .agg(
#         sin=("sin", "mean"),
#         cos=("cos", "mean"),
#         dY=("dY", "mean"),
#         dX=("dX", "mean"),
#         distTotal=("step_len", "sum"),
#         n_steps=("direction", "size"),
#     )
#     .reset_index()
# )

# # 2) Mahalanobis-based outlier detection per (order, density) group
# def mahalanobis_flags(df, cols=("dY", "dX"), threshold=2.5):
#     """
#     Returns a boolean mask of outliers and the distances.
#     Robust to singular covariance via pseudo-inverse.
#     """
#     X = df.loc[:, cols].to_numpy(dtype=float)
#     mu = X.mean(axis=0)
#     # covariance; rowvar=False for features in columns
#     cov = np.cov(X, rowvar=False)
#     # numerical safety: handle 1x1 or singular covariance
#     if X.shape[0] < 3 or not np.all(np.isfinite(cov)) or np.linalg.matrix_rank(cov) < len(cols):
#         inv_cov = np.linalg.pinv(cov)
#     else:
#         inv_cov = np.linalg.inv(cov)
#     diffs = X - mu
#     # Mahalanobis distance: sqrt( (x-mu)^T Sigma^{-1} (x-mu) )
#     d2 = np.einsum("ij,jk,ik->i", diffs, inv_cov, diffs)  # squared distances
#     d = np.sqrt(np.maximum(d2, 0.0))
#     outliers = d > float(threshold)
#     return outliers, d

# trial_features["mahalanobis_distance"] = np.nan
# trial_features["outlier"] = False

# for (o, d), grp in trial_features.groupby(["order", "density"], sort=False):
#     if len(grp) < 5:
#         # too few trials to estimate a stable covariance; skip outliering
#         trial_features.loc[grp.index, "outlier"] = False
#         trial_features.loc[grp.index, "mahalanobis_distance"] = 0.0
#         continue
#     flags, dists = mahalanobis_flags(grp, cols=("dY", "dX"), threshold=2.5)
#     trial_features.loc[grp.index, "outlier"] = flags
#     trial_features.loc[grp.index, "mahalanobis_distance"] = dists

# # 3) Keep only inlier trials in the rediscretized dataframe
# inlier_fnames = set(trial_features.loc[~trial_features["outlier"], "fname"])
# filtered_merged_df = merged_df[merged_df["fname"].isin(inlier_fnames)].copy()

# # 4) Rebuild the numeric animal_id after filtering (optional; mirrors your code)
# filtered_merged_df["unique_id"] = (
#     filtered_merged_df["order"].astype(str) + "|" +
#     filtered_merged_df["density"].astype(str) + "|" +
#     filtered_merged_df["fname"].astype(str)
# )
# id_map = {u: i for i, u in enumerate(filtered_merged_df["unique_id"].unique())}
# filtered_merged_df["animal_id"] = filtered_merged_df["unique_id"].map(id_map).astype(int)
# filtered_merged_df.drop(columns=["unique_id"], inplace=True)

# print("Before outlier removal:", merged_df["fname"].nunique(), "trials")
# print("After outlier removal:", filtered_merged_df["fname"].nunique(), "trials")
# print(filtered_merged_df.head())
# # -------------------------------------------------------------------------------

In [ ]:
# merged_df = filtered_merged_df.copy()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def _sorted_unique(series):
    vals = series.dropna().unique().tolist()
    try:
        return sorted(vals, key=float)
    except Exception:
        return sorted(vals)

# subplot ordering (top->bottom, left->right)
orders    = _sorted_unique(merged_df["order"])     # low at top, high at bottom (if ascending)
densities = _sorted_unique(merged_df["density"])   # low at left, high at right

print("Sorted orders:", orders)
print("Sorted densities:", densities)

n_rows = len(orders)
n_cols = len(densities)

plt.rcParams.update({
    "font.size": 8,
    "ytick.major.width": 1,
    "xtick.major.width": 1,
    "axes.linewidth": 1,
    "font.family": "Arial",
})

# -------------------------------------------------
# NEW 2D colormap: black->blue (density) + black->red (order) => deep purple
# -------------------------------------------------
def _lerp(a, b, t):
    return (1.0 - t) * np.asarray(a, float) + t * np.asarray(b, float)

def cell_color(order_idx, density_idx, n_rows, n_cols,
               tmin=0.10, tmax=0.95,
               blue=(0.10, 0.25, 1.00),
               red =(1.00, 0.15, 0.15),
               gamma=1.15,
               mix="max"):
    # row 0 = top (low order), bottom = high order
    to = (order_idx   / max(n_rows - 1, 1))
    td = (density_idx / max(n_cols - 1, 1))

    to = np.power(to, gamma)
    td = np.power(td, gamma)

    to = tmin + (tmax - tmin) * to
    td = tmin + (tmax - tmin) * td

    black = np.array([0.0, 0.0, 0.0], float)
    c_order = _lerp(black, red,  to)   # black -> red
    c_dens  = _lerp(black, blue, td)   # black -> blue

    if mix == "add":
        c = c_order + c_dens
    else:  # deep purple field (recommended)
        c = np.maximum(c_order, c_dens)

    return tuple(np.clip(c, 0, 1))

# -----------------------------
# Plot grid
# -----------------------------

# Figure: the matplotlib 'how' lives in locust_density_plots.py.
plots.plot_trajectory_grid(merged_df, orders, densities)


Sorted orders: [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
Sorted densities: [1, 4, 8, 16, 32, 64]


Saved figures to: figures/locust/trajectories_order_by_density_grid.pdf; figures/locust/trajectories_order_by_density_grid.png


In [ ]:
# --- Mean vectors per individual as lines (no axes; reference circle + 0° line; key matches panel colors) ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# In "step-length units": (dX, dY) / STEP_LENGTH
SCALE = 1 / STEP_LENGTH

def _sorted_unique(series):
    vals = series.dropna().unique().tolist()
    try:
        return sorted(vals, key=float)
    except Exception:
        return sorted(vals)

# -------------------------------------------------
# NEW 2D colormap: black->blue (density) + black->red (order) => deep purple
#   - density increases L->R  (blue channel contribution increases)
#   - order   increases T->B  (red channel contribution increases)
# -------------------------------------------------
def _lerp(a, b, t):
    return (1.0 - t) * np.asarray(a, float) + t * np.asarray(b, float)

def cell_color(order_idx, density_idx, n_rows, n_cols,
               tmin=0.10, tmax=0.95,
               blue=(0.10, 0.25, 1.00),
               red =(1.00, 0.15, 0.15),
               gamma=1.15,
               mix="max"):
    to = (order_idx   / max(n_rows - 1, 1))
    td = (density_idx / max(n_cols - 1, 1))

    to = np.power(to, gamma)
    td = np.power(td, gamma)

    to = tmin + (tmax - tmin) * to
    td = tmin + (tmax - tmin) * td

    black = np.array([0.0, 0.0, 0.0], float)
    c_order = _lerp(black, red,  to)   # black -> red
    c_dens  = _lerp(black, blue, td)   # black -> blue

    if mix == "add":
        c = c_order + c_dens
    else:  # deep purple field
        c = np.maximum(c_order, c_dens)

    return tuple(np.clip(c, 0, 1))

# 1) Compute mean vectors per (order, density, fname)
means_df = (
    merged_df
    .groupby(["order", "density", "fname"], as_index=False)[["dX", "dY"]]
    .mean()
)

if means_df.empty:
    raise RuntimeError("No mean vectors found. Check that merged_df has dX/dY and is not empty.")

# IMPORTANT: These define subplot ordering (top->bottom, left->right)
orders    = _sorted_unique(means_df["order"])     # ascending: low at top, high at bottom
densities = _sorted_unique(means_df["density"])   # ascending: low at left, high at right

n_rows = len(orders)
n_cols = len(densities)

# 2) Auto limits: symmetric around 0 using a 'nice' ceiling of global max |component|
max_abs_component = float(np.max(np.abs(means_df[["dX", "dY"]].to_numpy())))
pad = 1.10

def nice_ceil(v):
    if v <= 0:
        return 1.0
    v = v * pad
    exp = np.floor(np.log10(v))
    frac = v / (10 ** exp)
    for nice in (1.0, 2.0, 2.5, 5.0, 10.0):
        if frac <= nice:
            return nice * (10 ** exp)
    return 10.0 * (10 ** exp)

L = nice_ceil(max_abs_component * SCALE)
L = max(L, 1.15)  # ensure the radius-1 reference circle is always visible
XLIM = (-L, L)
YLIM = (-L, L)

# 3) Style
plt.rcParams.update({
    "font.size": 8,
    "axes.linewidth": 1,
    "font.family": "Arial",
})

# 4) Plot grid rows=order, cols=density

# Figure: the matplotlib 'how' lives in locust_density_plots.py.
plots.plot_mean_vector_grid(means_df, STEP_LENGTH)


Saved figures to: figures/locust/mean_vectors_order_by_density_grid.pdf; figures/locust/mean_vectors_order_by_density_grid.png


In [ ]:
# Synthetic schematic: density dots (left) + order trajectories (right)
# per (order, density) row. Self-contained, drawn by the sibling script.
plots.plot_order_density_trajectories()


Saved figures to: figures/locust/order_density_trajectories.pdf; figures/locust/order_density_trajectories.png


In [ ]:
# Synthetic 6x6 order-density grid (n_traj == density), 2D key on the
# right. Pure drawing, in locust_density_plots.py.
plots.plot_order_density_pairwise_trajectories()


Saved figures to: figures/locust/order_density_pairwise_trajectories_only.pdf; figures/locust/order_density_pairwise_trajectories_only.png


In [ ]:
# # --- Circular KDE per animal_id on the circle edge (no angular distortion) ---
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt

# # CONFIG
# ID_COL = 'animal_id'          # change to 'fname' if that's your per-individual column
# KAPPA = 20.0                  # von Mises concentration (bigger = smoother)
# N_ANG = 360                   # angular resolution for the curve
# THICKNESS = 0.25              # how far density can extend outward from r=1 (panel autoscale accounts for this)
# LINEWIDTH = 0.5               # curve width for each animal’s KDE
# BASE_CIRCLE_ALPHA = 0.25      # visibility of the reference unit circle

# # 0) Safety checks
# required = {'order','density',ID_COL,'dX','dY'}
# missing = required - set(merged_df.columns)
# if missing:
#     raise RuntimeError(f"merged_df is missing columns: {missing}")

# # 1) Compute heading angles (wrap in [-pi, pi))
# angles = np.arctan2(merged_df['dY'].to_numpy(), merged_df['dX'].to_numpy())
# merged_angles = merged_df.assign(theta=angles)

# # 2) Small helper for sorting
# def _sorted_unique(series):
#     vals = series.dropna().unique().tolist()
#     try:
#         return sorted(vals, key=float)
#     except Exception:
#         return sorted(vals)

# orders    = _sorted_unique(merged_angles['order'])
# densities = _sorted_unique(merged_angles['density'])

# # 3) von Mises circular KDE
# #    f(theta) = (1/n) * sum_i VM_kappa(theta - theta_i), VM_kappa(Δ) = exp(kappa*cos(Δ)) / (2π I0(kappa))
# def circular_kde_vm(sample_angles, thetas_eval, kappa=KAPPA):
#     sample_angles = np.asarray(sample_angles, dtype=float)
#     if sample_angles.size == 0:
#         return np.zeros_like(thetas_eval)
#     # pairwise differences (broadcast)
#     d = thetas_eval[:, None] - sample_angles[None, :]
#     num = np.exp(kappa * np.cos(d))
#     denom = 2.0 * np.pi * np.i0(kappa)  # I0 is the modified Bessel of first kind, order 0
#     f = num.sum(axis=1) / (denom * sample_angles.size)
#     return f

# thetas_eval = np.linspace(-np.pi, np.pi, N_ANG, endpoint=False)

# # 4) Shared limits/ticks (circle + outward thickness)
# L = 1.0 + THICKNESS + 0.05  # padding beyond max radius
# XLIM = (-L, L)
# YLIM = (-L, L)
# N_TICKS = 3
# XTICKS = np.linspace(-L, L, N_TICKS)
# YTICKS = np.linspace(-L, L, N_TICKS)

# print(f"Auto limits set to XLIM={XLIM}, YLIM={YLIM}")
# print(f"Auto ticks set to XTICKS={XTICKS}, YTICKS={YTICKS}")

# # 5) Plot grid rows=order, cols=density
# plt.rcParams.update({
#     'font.size': 8,
#     'ytick.major.width': 1,
#     'xtick.major.width': 1,
#     'axes.linewidth': 1,
#     'font.family': 'Arial'
# })

# n_rows = len(orders)
# n_cols = len(densities)

# fig, axes = plt.subplots(
#     n_rows, n_cols,
#     figsize=(n_cols * 1.6, n_rows * 1.6),
#     dpi=300,
#     squeeze=False
# )

# cmap = plt.cm.tab20

# for r, o in enumerate(orders):
#     for c, d in enumerate(densities):
#         ax = axes[r, c]
#         panel = merged_angles[(merged_angles['order'] == o) & (merged_angles['density'] == d)]

#         # empty panel?
#         if panel.empty:
#             ax.set_title(f"d={d} | o={o}", fontsize=8)
#             ax.set_xlim(*XLIM); ax.set_ylim(*YLIM)
#             ax.set_xticks(XTICKS); ax.set_yticks(YTICKS)
#             ax.set_aspect('equal')
#             ax.text(0.5, 0.5, "no data", ha='center', va='center',
#                     transform=ax.transAxes, alpha=0.6)
#             continue

#         # draw the base unit circle (r=1)
#         t = np.linspace(0, 2*np.pi, 512)
#         ax.plot(np.cos(t), np.sin(t), lw=0.8, alpha=BASE_CIRCLE_ALPHA, color='k')

#         # loop animals in this panel
#         animals = _sorted_unique(panel[ID_COL])
#         for i, aid in enumerate(animals):
#             angs = panel.loc[panel[ID_COL] == aid, 'theta'].to_numpy()

#             # KDE on the unit circle, then normalize per animal (for visual comparability within panel)
#             f = circular_kde_vm(angs, thetas_eval, kappa=KAPPA)
#             if np.allclose(f.max(), 0):
#                 continue
#             f_norm = f / f.max()  # scale peak to 1

#             # put density on the edge (r=1) and extend outward only by THICKNESS
#             r_curve = 0.5 + THICKNESS * f_norm
#             x = r_curve * np.cos(thetas_eval)
#             y = r_curve * np.sin(thetas_eval)

#             ax.plot(x, y, linewidth=LINEWIDTH, alpha=0.95, color=cmap(i / max(len(animals), 1)))

#         # aesthetics
#         ax.set_xlim(*XLIM); ax.set_ylim(*YLIM)
#         ax.set_xticks(XTICKS); ax.set_yticks(YTICKS)
#         ax.set_aspect('equal')
#         ax.set_title(f"d={d} | o={o}", fontsize=8)

# fig.suptitle(
#     "Per-animal circular KDEs by Order (rows) × Density (cols)\n"
#     "(density starts at r=1 and grows outward; no angular distortion)",
#     fontsize=9, y=1.02
# )
# fig.tight_layout()
# plt.show()


In [ ]:
# # --- Population circular KDE per condition (hierarchical bootstrap) with SEAMLESS WRAP on polar axes ---
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from matplotlib.patches import Circle
# from joblib import Parallel, delayed
# from tqdm.notebook import tqdm

# # =========================
# # Config
# # =========================
# ID_COL       = 'animal_id'   # change to 'fname' if needed
# COND_COLS    = ['order','density']
# KAPPA        = 30.0          # von Mises concentration
# N_POINTS     = 360           # angular resolution
# N_BOOT       = 1000
# CI_LO, CI_HI = 2.5, 97.5
# EDGE_R       = 1.0           # start density at the circle edge
# THICKNESS    = 0.35          # visual outward thickness at peak
# RIB_ALPHA    = 0.25
# MED_LINE_W   = 1.6

# # =========================
# # Utilities
# # =========================
# def _sorted_unique(series):
#     vals = series.dropna().unique().tolist()
#     try:
#         return sorted(vals, key=float)
#     except Exception:
#         return sorted(vals)

# def bootstrap_summary_stats(bootstrap_samples, axis=0, lower=2.5, upper=97.5):
#     p = np.percentile(bootstrap_samples, [lower, 50, upper], axis=axis)
#     lo, med, hi = p
#     return {'median': med, 'lower_bounds': lo, 'upper_bounds': hi}

# def hierarchical_bootstrap(data, labels, compute_func, n_bootstraps=10000, n_jobs=-1, random_state=None):
#     data = np.asarray(data); labels = np.asarray(labels)
#     rng = np.random.default_rng(random_state)
#     uniq, inv = np.unique(labels, return_inverse=True)
#     buckets = [np.flatnonzero(inv == i) for i in range(len(uniq))]
#     nL = len(uniq)

#     def one(seed):
#         r = np.random.default_rng(seed)
#         Lids = r.integers(0, nL, size=nL, endpoint=False)
#         perL = []
#         for lid in Lids:
#             idx = buckets[lid]
#             boot_idx = idx[r.integers(0, len(idx), size=len(idx), endpoint=False)]
#             perL.append(compute_func(data[boot_idx]))
#         return np.mean(np.asarray(perL), axis=0)

#     seeds = rng.integers(0, np.iinfo(np.int32).max, size=n_bootstraps, endpoint=False)
#     out = Parallel(n_jobs=n_jobs)(delayed(one)(int(s)) for s in tqdm(seeds, desc="Hierarchical bootstrapping"))
#     return np.asarray(out)

# def von_mises_kde(data, pts, kappa):
#     c = 1.0 / (2.0 * np.pi * np.i0(kappa))
#     return c * np.exp(kappa * np.cos(pts[:, None] - data[None, :])).mean(axis=1)

# def bootstrap_von_mises_kde(data, labels, kappa, n_bootstraps=4000, n_points=360):
#     data = (data + np.pi) % (2*np.pi) - np.pi
#     ang = np.linspace(-np.pi, np.pi, n_points, endpoint=False)
#     def comp(samp): return von_mises_kde(samp, ang, kappa)
#     boots = hierarchical_bootstrap(data, labels, comp, n_bootstraps=n_bootstraps, n_jobs=-1)
#     return {'angles': ang, 'bootstrap_kdes': boots}

# # --- wrap helpers to avoid seam gaps on polar ---
# def _to_0_2pi(theta):
#     """Map [-pi,pi) -> [0,2pi) and return sorted angles + order index."""
#     t = (theta + 2*np.pi) % (2*np.pi)
#     order = np.argsort(t)
#     return t[order], order

# def _wrap_curve(theta, r):
#     """
#     Ensure continuity across seam for plot() by appending the first point at +2π.
#     theta must already be in [0, 2π) **and sorted**.
#     """
#     return np.r_[theta, theta[0] + 2*np.pi], np.r_[r, r[0]]

# def _wrap_ribbon(theta, r_lo, r_hi):
#     """
#     Produce a single closed path (theta_hi ++ theta_lo[::-1], same for r) for fill_between on polar.
#     Inputs theta in [0,2π) sorted. We also append seam points.
#     """
#     th_hi, r_hi2 = _wrap_curve(theta, r_hi)
#     th_lo, r_lo2 = _wrap_curve(theta, r_lo)
#     th = np.r_[th_hi, th_lo[::-1]]
#     rr = np.r_[r_hi2, r_lo2[::-1]]
#     return th, rr

# # =========================
# # Prepare data
# # =========================
# # ensure we have directions
# if 'direction' not in merged_df.columns:
#     if {'dX','dY'}.issubset(merged_df.columns):
#         merged_df = merged_df.assign(direction=np.arctan2(merged_df['dY'].to_numpy(),
#                                                           merged_df['dX'].to_numpy()))
#     else:
#         raise RuntimeError("Provide 'direction' or ('dX','dY') in merged_df.")

# if ID_COL not in merged_df.columns:
#     ID_COL = 'fname' if 'fname' in merged_df.columns else ID_COL
#     if ID_COL not in merged_df.columns:
#         raise RuntimeError(f"Could not find ID column ('animal_id' or 'fname').")

# for col in COND_COLS:
#     if col not in merged_df.columns:
#         raise RuntimeError(f"Missing condition column '{col}' in merged_df.")

# orders    = _sorted_unique(merged_df[COND_COLS[0]])
# densities = _sorted_unique(merged_df[COND_COLS[1]])

# # =========================
# # Figure scaffold (polar grid)
# # =========================
# plt.rcParams['ytick.major.width'] = 2
# plt.rcParams['xtick.major.width'] = 2
# plt.rcParams['axes.linewidth'] = 2
# plt.rcParams['font.family'] = 'Arial'
# plt.rcParams.update({'font.size': 8})

# n_rows, n_cols = len(orders), len(densities)
# fig, axes = plt.subplots(n_rows, n_cols, subplot_kw={'projection': 'polar'},
#                          figsize=(n_cols*2.2, n_rows*2.2), dpi=300, squeeze=False)

# # ticks at [-π, -π/2, 0, π/2, π] but mapped into [0, 2π)
# tick_angles = np.array([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
# tick_angles_02 = (tick_angles + 2*np.pi) % (2*np.pi)
# tick_labels = [r"", r"$-\frac{1}{2}\pi$", r"$0$", r"$\frac{1}{2}\pi$", r"$\pm\pi$"]

# for r_i, o in enumerate(orders):
#     for c_i, d in enumerate(densities):
#         ax = axes[r_i, c_i]
#         panel = merged_df[(merged_df[COND_COLS[0]] == o) & (merged_df[COND_COLS[1]] == d)]

#         # base styling
#         ax.set_theta_zero_location("E")   # 0 at +x axis (like Cartesian), optional
#         ax.set_theta_direction(-1)        # increase clockwise to match many compass-like plots, optional
#         ax.set_xticks(tick_angles_02)
#         ax.set_xticklabels(tick_labels, fontsize=10)
#         ax.set_yticks([])
#         ax.spines['polar'].set_visible(False)
#         ax.set_title(f"d={d} | o={o}", va='bottom', fontsize=8)

#         if panel.empty:
#             # draw just the edge circle
#             ax.plot(np.linspace(0, 2*np.pi, 361, endpoint=True),
#                     np.full(361, EDGE_R), color="black", linewidth=0.5)
#             ax.text(np.deg2rad(90), EDGE_R+0.1, "no data", ha='center', va='center', alpha=0.6)
#             continue

#         # outer reference edge (r=1)
#         ax.plot(np.linspace(0, 2*np.pi, 361, endpoint=True),
#                 np.full(361, EDGE_R), color="black", linewidth=0.5)

#         # data & labels for this condition
#         th = panel['direction'].to_numpy()
#         ids = panel[ID_COL].to_numpy()

#         # hierarchical bootstrap KDE
#         boot = bootstrap_von_mises_kde(th, ids, kappa=KAPPA, n_bootstraps=N_BOOT, n_points=N_POINTS)
#         summ = bootstrap_summary_stats(boot['bootstrap_kdes'], axis=0, lower=CI_LO, upper=CI_HI)
#         ang = boot['angles']                          # in [-π, π)
#         med, lo, hi = summ['median'], summ['lower_bounds'], summ['upper_bounds']

#         # panel-wise normalization so max(hi) reaches THICKNESS
#         scale = np.max(hi) if np.max(hi) > 0 else 1.0
#         med_n, lo_n, hi_n = med/scale, lo/scale, hi/scale

#         # radii (start at EDGE_R and grow outward)
#         r_med = EDGE_R + THICKNESS * med_n
#         r_lo  = EDGE_R + THICKNESS * lo_n
#         r_hi  = EDGE_R + THICKNESS * hi_n

#         # map to [0,2π), sort, and build wrapped paths
#         ang02, order = _to_0_2pi(ang)
#         r_med_s = r_med[order]; r_lo_s = r_lo[order]; r_hi_s = r_hi[order]

#         th_curve, r_curve = _wrap_curve(ang02, r_med_s)
#         th_ribbon, r_ribbon = _wrap_ribbon(ang02, r_lo_s, r_hi_s)

#         # draw CI ribbon and median line (seamless)
#         ax.fill(th_ribbon, r_ribbon, color='tab:blue', alpha=RIB_ALPHA, linewidth=0)
#         ax.plot(th_curve, r_curve, color='tab:blue', linewidth=MED_LINE_W)

#         # radial limit
#         ax.set_rlim(0, EDGE_R + THICKNESS * 1.2)

# # (Optional) Shared legend proxy (one panel)
# # You can build a custom legend like your example using handler_map/customer_handler if you have it defined.
# # Here’s a minimal version without custom handlers:
# from matplotlib.lines import Line2D
# legend_handles = [
#     Line2D([0],[0], color='tab:blue', lw=MED_LINE_W, label='KDE (median)'),
#     Line2D([0],[0], color='tab:blue', lw=8, alpha=RIB_ALPHA, label='KDE 95% CI')
# ]
# axes[0,0].legend(handles=legend_handles, bbox_to_anchor=(1, 1), frameon=False, loc="center left")

# fig.suptitle('Population-Level Circular KDE per Condition (hierarchical bootstrap)\n'
#              'Density starts at r=1 and grows outward; curves/ribbons wrap seamlessly',
#              y=1.02, fontsize=9)
# plt.tight_layout()
# plt.show()


In [ ]:
# Per-trial mean movement vector + heading -> angles_df (the table the
# von Mises model cell fits and the alignment figures draw).
angles_df = dens.mean_vector_angles(merged_df)

print(angles_df.head())


In [ ]:
from math import pi
import numpy as np
import pandas as pd
import bambi as bmb

# Links & priors from unwrap (as in your example)
from unwrap.prior import CircularUniformPrior
from unwrap.link.bambi import circular_modulo, softplus
from unwrap.link import inverse_softplus

# ----------------------------
# Transform helpers (your defs)
# ----------------------------
def normalize(x):
    return x * 2.0 - 1.0

def log2_scale(x, min_size=1., max_size=64.):
    return np.log2(x / min_size) / np.log2(max_size / min_size)

def inv_log2_scale(x, min_size=1., max_size=64.):
    inv = x * np.log2(max_size / min_size)
    inv = 2**inv
    inv *= min_size
    return inv

def inv_normalize(x):
    return (x + 1.0) / 2.0

def exp_scale(x, min_size=1., max_size=64.):
    return (x - min_size) / (max_size - min_size)

# ----------------------------
# Build modeling table
# ----------------------------
# angles_df must have: ['order','density','fname','angle_rad'] from the previous step
df = angles_df.copy()

# Make sure y is wrapped to (-pi, pi]
df["y"] = ((df["angle_rad"] + np.pi) % (2 * np.pi)) - np.pi

# Choose scaling bounds for density (adjust to your experiment if needed)
DENS_MIN, DENS_MAX = 1.0, 64.0  # <- change these to your actual density range

# Transform predictors
# df["t_density"] = normalize(exp_scale(df["density"].astype(float), DENS_MIN, DENS_MAX))
df["t_density"] = normalize(log2_scale(df["density"].astype(float), DENS_MIN, DENS_MAX))
df["t_order"]   = normalize(df["order"].astype(float))

# (Optional) sanity checks
assert df["t_density"].between(-1, 1).all()
assert df["t_order"].between(-1, 1).all()

# ----------------------------
# Specify the model
# ----------------------------
# μ depends on both predictors and their interaction.
# κ depends on both predictors (you can add the interaction if you need it).
formula = bmb.Formula("y ~ t_density * t_order",
                      "kappa ~ t_density * t_order"
                     )

priors = {
    # μ parameters (circular)
    "Intercept":  bmb.Prior("VonMises", mu=0.0, kappa=1e-6),                      # U(-pi, pi]
    "t_density": bmb.Prior("Normal", mu=0.0, sigma=0.5*pi),   # slope priors on μ
    "t_order":   bmb.Prior("Normal", mu=0.0, sigma=0.5*pi),
    "t_density:t_order": bmb.Prior("Normal", mu=0.0, sigma=0.5*pi),

    # κ parameters (on pre-link scale)
    "kappa": {
        "Intercept": bmb.Prior("Normal", mu=inverse_softplus(2.0), sigma=1.0),  # baseline κ≈1
        "t_density": bmb.Prior("Normal", mu=0.0, sigma=1.0),
        "t_order":   bmb.Prior("Normal", mu=0.0, sigma=1.0),
        "t_density:t_order":   bmb.Prior("Normal", mu=0.0, sigma=1.0),
    },
}

model = bmb.Model(
    formula=formula,
    data=df,
    family="vonmises",
    priors=priors,
    link={"mu": circular_modulo, "kappa": softplus},
    center_predictors=False,
)

# Fit
idata = model.fit(random_seed=123, idata_kwargs=dict(log_likelihood = True))

# Quick summary
print(model)
print(idata)
# bmb.summary(idata)  # or arviz.summary(idata)


Initializing NUTS using jitter+adapt_diag...


/home/jake/.venvs/unwrap-env/lib/python3.12/site-packages/pytensor/link/c/cmodule.py:2986: UserWarning: PyTensor could not link to a BLAS installation. Operations that might benefit from BLAS will be severely degraded.
This usually happens when PyTensor is installed via pip. We recommend it be installed via conda/mamba/pixi instead.
Alternatively, you can use an experimental backend such as Numba or JAX that perform their own BLAS optimizations, by setting `pytensor.config.mode == 'NUMBA'` or passing `mode='NUMBA'` when compiling a PyTensor function.
For more options and details see https://pytensor.readthedocs.io/en/latest/troubleshooting.html#how-do-i-configure-test-my-blas-library
  warnings.warn(


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [Intercept, t_density, t_order, t_density:t_order, kappa_Intercept, kappa_t_density, kappa_t_order, kappa_t_density:t_order]


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 12 seconds.


       Formula: y ~ t_density * t_order
                kappa ~ t_density * t_order
        Family: vonmises
          Link: mu = circular_modulo
                kappa = softplus
  Observations: 440
        Priors: 
    target = mu
        Common-level effects
            Intercept ~ VonMises(mu: 0.0, kappa: 0.0)
            t_density ~ Normal(mu: 0.0, sigma: 1.5708)
            t_order ~ Normal(mu: 0.0, sigma: 1.5708)
            t_density:t_order ~ Normal(mu: 0.0, sigma: 1.5708)
    target = kappa
        Common-level effects
            kappa_Intercept ~ Normal(mu: 1.8546, sigma: 1.0)
            kappa_t_density ~ Normal(mu: 0.0, sigma: 1.0)
            kappa_t_order ~ Normal(mu: 0.0, sigma: 1.0)
            kappa_t_density:t_order ~ Normal(mu: 0.0, sigma: 1.0)
------
* To see a plot of the priors call the .plot_priors() method.
* To see a summary or plot of the posterior pass the object returned by .fit() to az.summary() or az.plot_trace()
Inference data with groups:
	> posterior
	

In [ ]:
import arviz as az
az.summary(idata, circ_var_names=["Intercept"])

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
Intercept,0.045,0.080,-0.115,0.183,0.002,0.001,1962.0,2262.0,1.0
t_density,0.071,0.115,-0.147,0.282,0.003,0.002,2075.0,2404.0,1.0
t_order,-0.119,0.109,-0.330,0.077,0.002,0.002,1949.0,2482.0,1.0
t_density:t_order,0.143,0.158,-0.160,0.434,0.003,0.002,2200.0,2442.0,1.0
kappa_Intercept,1.370,0.169,1.065,1.694,0.003,0.003,3406.0,2882.0,1.0
kappa_t_density,0.811,0.240,0.356,1.262,0.004,0.004,4171.0,2873.0,1.0
kappa_t_order,2.525,0.281,1.980,3.024,0.005,0.004,3364.0,2766.0,1.0
kappa_t_density:t_order,1.009,0.394,0.296,1.770,0.006,0.006,3733.0,3030.0,1.0


In [ ]:
# --- Predict directional alignment = A(kappa) * cos(mu) and replicate two panels ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arviz as az
from scipy.special import i0, i1  # A(kappa) = I1/I0

# same transforms you used in the model
from math import pi
def normalize(x): return x * 2.0 - 1.0
def log2_scale(x, min_size=1., max_size=64.):
    return np.log2(x / min_size) / np.log2(max_size / min_size)

DENS_MIN, DENS_MAX = 1.0, 64.0  # match training bounds

def softplus(x):
    return np.log1p(np.exp(x))

def circular_modulo(x):
    return (x + np.pi) % (2 * np.pi) - np.pi

def A_of_kappa(kappa):
    return i1(kappa) / i0(kappa)

def posterior_linpred(idata, prefix, X_df):
    post = idata.posterior
    if prefix:
        coefs = [v for v in post.data_vars if v.startswith(prefix)]
        key_intercept = f"{prefix}Intercept"
    else:
        coefs = [v for v in post.data_vars if (not v.startswith("kappa_"))]
        key_intercept = "Intercept"

    if key_intercept not in post:
        raise ValueError(f"Intercept for prefix '{prefix}' not found in posterior.")

    def flat(name):
        return post[name].stack(sample=("chain", "draw")).values

    n = len(X_df)
    eta = np.tile(flat(key_intercept)[:, None], (1, n))

    for coef in coefs:
        if coef == key_intercept:
            continue
        col = coef.replace(prefix, "")
        if col in X_df.columns:
            eta += flat(coef)[:, None] * X_df[col].to_numpy()[None, :]

    return eta

def make_X(order_vals, density_vals, idata):
    X = pd.DataFrame({"order": order_vals, "density": density_vals})
    X["t_order"]   = normalize(X["order"].astype(float))
    X["t_density"] = normalize(log2_scale(X["density"].astype(float), DENS_MIN, DENS_MAX))
    if ("t_density:t_order" in idata.posterior.data_vars or
        "kappa_t_density:t_order" in idata.posterior.data_vars):
        X["t_density:t_order"] = X["t_density"] * X["t_order"]
    return X

def posterior_alignment(idata, X):
    eta_mu    = posterior_linpred(idata, prefix="",       X_df=X)
    eta_kappa = posterior_linpred(idata, prefix="kappa_", X_df=X)

    mu    = circular_modulo(eta_mu)
    kappa = softplus(eta_kappa)

    A = A_of_kappa(kappa)
    align = A * np.cos(mu)

    mean = align.mean(axis=0)
    hdi68 = az.hdi(align, hdi_prob=0.68, skipna=True)
    return {"mean": mean, "low": hdi68[:, 0], "high": hdi68[:, 1]}

# ---- prediction grids ----
# DESCENDING so legends list largest -> smallest (top -> bottom)
dens_levels  = np.sort(merged_df["density"].unique())[::-1]
order_levels = np.sort(merged_df["order"].unique())[::-1]

order_grid   = np.linspace(order_levels.min(), order_levels.max(), 50)
density_grid = np.geomspace(max(dens_levels.min(), 1e-6), dens_levels.max(), 50)

# Panel (i): vs Order, colored by Density
pred_i = {}
for d in dens_levels:
    X = make_X(order_grid, np.full_like(order_grid, d, dtype=float), idata)
    keep_cols = ["t_order", "t_density"] + (["t_density:t_order"] if "t_density:t_order" in X.columns else [])
    pred_i[d] = posterior_alignment(idata, X[keep_cols])

# Panel (ii): vs Density, colored by Order
pred_ii = {}
for o in order_levels:
    X = make_X(np.full_like(density_grid, o, dtype=float), density_grid, idata)
    keep_cols = ["t_order", "t_density"] + (["t_density:t_order"] if "t_density:t_order" in X.columns else [])
    pred_ii[o] = posterior_alignment(idata, X[keep_cols])

# ---- plot with legends (same style) ----
plt.rcParams.update({
    "font.size": 8, "ytick.major.width": 1, "xtick.major.width": 1,
    "axes.linewidth": 1, "font.family": "Arial"
})

# Figure: the matplotlib 'how' lives in locust_density_plots.py.
plots.plot_alignment_prediction(
    order_grid, density_grid, dens_levels, order_levels, pred_i, pred_ii
)


/tmp/ipykernel_3881822/2547916176.py:72: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi68 = az.hdi(align, hdi_prob=0.68, skipna=True)
/tmp/ipykernel_3881822/2547916176.py:72: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi68 = az.hdi(align, hdi_prob=0.68, skipna=True)
/tmp/ipykernel_3881822/2547916176.py:72: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi68 = az.hdi(align, hdi_prob=0.68, skipna=True)


/tmp/ipykernel_3881822/2547916176.py:72: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi68 = az.hdi(align, hdi_prob=0.68, skipna=True)
/tmp/ipykernel_3881822/2547916176.py:72: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi68 = az.hdi(align, hdi_prob=0.68, skipna=True)
/tmp/ipykernel_3881822/2547916176.py:72: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi68 = az.hdi(align, hdi_prob=0.68, skipna=True)


/tmp/ipykernel_3881822/2547916176.py:72: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi68 = az.hdi(align, hdi_prob=0.68, skipna=True)
/tmp/ipykernel_3881822/2547916176.py:72: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi68 = az.hdi(align, hdi_prob=0.68, skipna=True)
/tmp/ipykernel_3881822/2547916176.py:72: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi68 = az.hdi(align, hdi_prob=0.68, skipna=True)


/tmp/ipykernel_3881822/2547916176.py:72: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi68 = az.hdi(align, hdi_prob=0.68, skipna=True)
/tmp/ipykernel_3881822/2547916176.py:72: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi68 = az.hdi(align, hdi_prob=0.68, skipna=True)
/tmp/ipykernel_3881822/2547916176.py:72: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi68 = az.hdi(align, hdi_prob=0.68, skipna=True)


In [ ]:
# # --- put these near the top with your imports ---
# from math import pi
# import numpy as np
# import pandas as pd
# import arviz as az
# import matplotlib.pyplot as plt
# import bambi as bmb

# from unwrap.prior import CircularUniformPrior
# from unwrap.link.bambi import circular_modulo, softplus
# from unwrap.link import inverse_softplus

# # -----------------------------
# # κ model set (your exact names)
# # -----------------------------
# KAPPA_MODELS = [
#     ("Intercept-only", []),
#     ("Density", ["t_density"]),
#     ("Order", ["t_order"]),
#     ("Density + Order", ["t_density", "t_order"]),
#     ("Density × Order", ["t_density", "t_order", "t_density:t_order"]),
# ]

# # -----------------------------
# # Build formula / priors helpers
# # -----------------------------
# def build_formula(kappa_terms):
#     # μ is ALWAYS intercept-only
#     mu_rhs = "t_density * t_order"
#     k_rhs = " + ".join(kappa_terms) if kappa_terms else "1"
#     return bmb.Formula(f"y ~ {mu_rhs}", f"kappa ~ {k_rhs}")

# # Prior templates
# # μ intercept prior: circular uniform
# # κ priors live on the pre-link (inverse-softplus) scale
# K_INTERCEPT_PRIOR = bmb.Prior("Normal", mu=inverse_softplus(1.0), sigma=1.0)
# K_SLOPE_PRIOR     = bmb.Prior("Normal", mu=0.0, sigma=1.0)

# def build_priors(kappa_terms):
#     pri = {}

#     # μ (intercept-only)
#     pri["Intercept"] = CircularUniformPrior()

#     # κ (intercept + optional terms)
#     pri["kappa"] = {"Intercept": K_INTERCEPT_PRIOR}
#     for term in kappa_terms:
#         if term == "1":
#             continue
#         pri["kappa"][term] = K_SLOPE_PRIOR

#     return pri

# def _grab(obj, *names):
#     """Robustly extract fields from arviz ELPD objects across versions."""
#     for n in names:
#         if hasattr(obj, n):
#             return getattr(obj, n)
#     for n in names:
#         try:
#             return obj[n]
#         except Exception:
#             pass
#     return None

# # ---------------------------------------
# # Fit + PSIS-LOO for κ-only model sweep
# # ---------------------------------------
# results = []
# models  = []
# idatas  = []

# for pretty_name, kappa_terms in KAPPA_MODELS:
#     print(f"\n=== Fitting {pretty_name} ===")

#     formula = build_formula(kappa_terms)
#     priors  = build_priors(kappa_terms)

#     mdl = bmb.Model(
#         formula=formula,
#         data=df,                 # assumes df already exists
#         family="vonmises",
#         priors=priors,
#         link={"mu": circular_modulo, "kappa": softplus},
#     )

#     fit_idata = mdl.fit(
#         draws=1000,
#         chains=4,
#         target_accept=0.9,
#         idata_kwargs={"log_likelihood": True},
#     )

#     loo_res = az.loo(fit_idata)

#     elpd = _grab(loo_res, "elpd_loo", "elpd")
#     se   = _grab(loo_res, "elpd_loo_se", "se", "elpd_se")
#     p_eff= _grab(loo_res, "p_loo", "p_eff")

#     results.append({
#         "pretty_name": pretty_name,
#         "kappa_terms": kappa_terms,
#         "elpd": float(elpd) if elpd is not None else float("nan"),
#         "se": float(se) if se is not None else float("nan"),
#         "p_eff": float(p_eff) if p_eff is not None else float("nan"),
#     })

#     models.append(mdl)
#     idatas.append(fit_idata)

# # Results table (rank by ELPD; higher is better)
# res_df = pd.DataFrame(results).sort_values("elpd", ascending=False).reset_index(drop=True)

# print("\n=== Model ranking by ELPD (higher is better) ===")
# print(res_df[["pretty_name", "elpd", "se", "p_eff"]].to_string(index=False))

# # Save results (optional)
# res_df.to_parquet("kappa_only_model_sweep_results.parquet", index=False)


## Model selection (individual-level, PSIS-LOO)

The sweep below fits the five candidate specifications to the
one-observation-per-animal screening table and ranks them by $\Delta$ELPD
relative to the best (next cell):

- **Density $\times$ Order** is the best specification, and **Order** alone
  is within a couple of $\Delta$ELPD units of it (small relative to its SE);
- **Density** alone is the weakest single-predictor model
  ($\Delta$ELPD $\approx$ −76).

This supports the **order-dominant** interpretation: the directional
structure of neighbor motion carries the bulk of the information about the
focal locust's alignment, and visual density adds at most a secondary effect
once order is included. The winning interaction specification is then fitted
to the full step-level data.


In [ ]:
from math import pi
import numpy as np
import pandas as pd
import arviz as az
import bambi as bmb

from unwrap.prior import CircularUniformPrior
from unwrap.link.bambi import circular_modulo, softplus
from unwrap.link import inverse_softplus

# -----------------------------
# Model set (same predictors for μ and κ)
# -----------------------------
MODELS = [
    ("Intercept-only", []),
    ("Density", ["t_density"]),
    ("Order", ["t_order"]),
    ("Density + Order", ["t_density", "t_order"]),
    ("Density × Order", ["t_density", "t_order", "t_density:t_order"]),
]

def rhs_from_terms(terms):
    return "1" if not terms else ("1 + " + " + ".join(terms))

def build_formulas(terms):
    mu_rhs = rhs_from_terms(terms)
    k_rhs  = rhs_from_terms(terms)
    mu_formula = f"y ~ {mu_rhs}"
    k_formula  = f"kappa ~ {k_rhs}"
    return mu_formula, k_formula, bmb.Formula(mu_formula, k_formula)

# Priors
MU_INTERCEPT_PRIOR = CircularUniformPrior()
MU_SLOPE_PRIOR     = bmb.Prior("Normal", mu=0.0, sigma=0.5 * np.pi)

K_INTERCEPT_PRIOR  = bmb.Prior("Normal", mu=inverse_softplus(2.0), sigma=1.0)
K_SLOPE_PRIOR      = bmb.Prior("Normal", mu=0.0, sigma=1.0)

def build_priors(terms):
    pri = {}
    pri["Intercept"] = MU_INTERCEPT_PRIOR
    for term in (terms or []):
        pri[term] = MU_SLOPE_PRIOR

    pri["kappa"] = {"Intercept": K_INTERCEPT_PRIOR}
    for term in (terms or []):
        pri["kappa"][term] = K_SLOPE_PRIOR
    return pri

def elpd_diff_from_pointwise(model_loo_i, base_loo_i):
    diff_i = (np.asarray(model_loo_i) - np.asarray(base_loo_i)).reshape(-1)
    n = diff_i.size
    elpd_diff = float(diff_i.sum())
    se_diff = float(np.sqrt(n * np.var(diff_i, ddof=1))) if n > 1 else float("nan")
    return elpd_diff, se_diff

# ---------------------------------------
# Fit + pointwise LOO storage
# ---------------------------------------
meta_rows = []
loo_i_by_name = {}

for pretty_name, terms in MODELS:
    print(f"\n=== Fitting {pretty_name} ===")

    mu_str, k_str, formula = build_formulas(terms)
    priors = build_priors(terms)

    mdl = bmb.Model(
        formula=formula,
        data=df,  # assumes df exists
        family="vonmises",
        priors=priors,
        link={"mu": circular_modulo, "kappa": softplus},
    )

    fit_idata = mdl.fit(
        draws=1000,
        chains=4,
        target_accept=0.9,
        random_seed=123,
        idata_kwargs={"log_likelihood": True},
    )

    loo = az.loo(fit_idata, pointwise=True)

    loo_i = np.asarray(loo.loo_i).reshape(-1)
    loo_i_by_name[pretty_name] = loo_i

    meta_rows.append({
        "pretty_name": pretty_name,
        "terms": terms,
        "mu_formula": mu_str,
        "kappa_formula": k_str,
        "elpd": float(loo.elpd_loo),
        "se_elpd": float(loo.se),
        "p_eff": float(loo.p_loo),
        "pareto_k_max": float(np.max(np.asarray(loo.pareto_k))),
    })

meta_df = pd.DataFrame(meta_rows)

# ---------------------------------------
# Manual pointwise ΔELPD vs BEST (highest elpd)
# ΔELPD = model - best  (best = 0, others <= 0)
# ---------------------------------------
best_name = meta_df.loc[meta_df["elpd"].idxmax(), "pretty_name"]
base_loo_i = loo_i_by_name[best_name]

meta_df["best_model"] = best_name

deltas = []
dse = []
for name in meta_df["pretty_name"]:
    d, se = elpd_diff_from_pointwise(loo_i_by_name[name], base_loo_i)
    deltas.append(d)
    dse.append(se)

meta_df["delta_elpd"] = deltas
meta_df["se_delta_elpd"] = dse

# sort best-first (delta 0 at top)
meta_df = meta_df.sort_values("delta_elpd", ascending=False).reset_index(drop=True)

OUT_PATH = "mu_kappa_matched_model_sweep_pointwise_delta.parquet"
meta_df.to_parquet(OUT_PATH, index=False)
print(f"\nBest model: {best_name}")
print(f"Saved: {OUT_PATH}")

meta_df


Initializing NUTS using jitter+adapt_diag...



=== Fitting Intercept-only ===


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [Intercept, kappa_Intercept]


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 3 seconds.


...

Multiprocess sampling (4 chains in 4 jobs)


NUTS: [Intercept, t_density, t_order, t_density:t_order, kappa_Intercept, kappa_t_density, kappa_t_order, kappa_t_density:t_order]


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 13 seconds.



Best model: Density × Order
Saved: mu_kappa_matched_model_sweep_pointwise_delta.parquet


,pretty_name,terms,mu_formula,kappa_formula,elpd,se_elpd,p_eff,pareto_k_max,best_model,delta_elpd,se_delta_elpd
0,Density × Order,"[t_density, t_order, t_density:t_order]",y ~ 1 + t_density + t_order + t_density:t_order,kappa ~ 1 + t_density + t_order + t_density:t_...,-590.520947,17.510507,7.862080,0.317989,Density × Order,0.000000,0.000000
1,Density + Order,"[t_density, t_order]",y ~ 1 + t_density + t_order,kappa ~ 1 + t_density + t_order,-592.963440,17.448913,6.386673,0.226814,Density × Order,-2.442493,2.612576
2,Order,[t_order],y ~ 1 + t_order,kappa ~ 1 + t_order,-599.800766,17.961610,4.167510,0.102140,Density × Order,-9.279819,5.739002
3,Density,[t_density],y ~ 1 + t_density,kappa ~ 1 + t_density,-666.417847,16.721787,4.231966,0.131385,Density × Order,-75.896900,11.557480
4,Intercept-only,[],y ~ 1,kappa ~ 1,-669.203208,16.487420,2.050087,0.005632,Density × Order,-78.682261,11.612378


In [ ]:
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import textwrap

RES_PATH = "mu_kappa_matched_model_sweep_pointwise_delta.parquet"
res_df = pd.read_parquet(RES_PATH)

def wrap_label(s, width=40):
    return textwrap.fill(str(s), width=width)

def _as_list(x):
    if x is None:
        return []
    if isinstance(x, (list, tuple)):
        return list(x)
    if isinstance(x, np.ndarray):
        return x.ravel().tolist()
    if isinstance(x, str):
        try:
            v = ast.literal_eval(x)
            if isinstance(v, np.ndarray):
                return v.ravel().tolist()
            if isinstance(v, (list, tuple)):
                return list(v)
        except Exception:
            return [x]
        return [x]
    try:
        if pd.isna(x):
            return []
    except Exception:
        pass
    return [str(x)]

res_df["has_order"] = res_df["terms"].apply(lambda t: any("t_order" in s for s in _as_list(t)))

# already sorted best-first in sweep, but keep safe:
res_df = res_df.sort_values("delta_elpd", ascending=False).reset_index(drop=True)

y = np.arange(len(res_df))
y_labels = [wrap_label(n, 40) for n in res_df["pretty_name"]]

g_incl = res_df[res_df["has_order"]].copy()
g_excl = res_df[~res_df["has_order"]].copy()
g_incl["y"] = g_incl.index.to_numpy()
g_excl["y"] = g_excl.index.to_numpy()

use_se = "se_delta_elpd" in res_df.columns and (not res_df["se_delta_elpd"].isna().all())

# Figure: the matplotlib 'how' lives in locust_density_plots.py.
plots.plot_model_selection(res_df, use_se)

print(res_df[["pretty_name", "elpd", "delta_elpd", "se_delta_elpd", "pareto_k_max", "best_model"]].to_string(index=False))


Saved figures to: figures/locust/order_density_model_selection.pdf; figures/locust/order_density_model_selection.png


    pretty_name        elpd  delta_elpd  se_delta_elpd  pareto_k_max      best_model
Density × Order -590.520947    0.000000       0.000000      0.317989 Density × Order
Density + Order -592.963440   -2.442493       2.612576      0.226814 Density × Order
          Order -599.800766   -9.279819       5.739002      0.102140 Density × Order
        Density -666.417847  -75.896900      11.557480      0.131385 Density × Order
 Intercept-only -669.203208  -78.682261      11.612378      0.005632 Density × Order


## Final hierarchical fit

The step-level model with a **random intercept for the concentration
($\kappa$) only**:

$$
y_t \sim \mathcal{V}(\mu(\mathbf{x}_i),\ \kappa(\mathbf{x}_i)),
\qquad
\eta_\mu = b_0 + b_d \tilde{d}_i + b_o \tilde{o}_i + b_{do}\,\tilde{d}_i\tilde{o}_i,
\qquad
\eta_\kappa = a_0 + a_d \tilde{d}_i + a_o \tilde{o}_i + a_{do}\,\tilde{d}_i\tilde{o}_i + v_i,
\qquad
v_i \sim \mathcal{N}(0, \sigma^2),
$$

with the `unwrap` circular-modulo link on $\mu$ (linear predictors are
mapped onto the circle at likelihood time) and the softplus link on
$\kappa > 0$. Each animal experienced exactly one order-density
condition, so order and density vary **between animals only** and no
random slope is identifiable. The per-animal *direction* (\mu) intercept is
likewise weakly identifiable on this design (10% of animals have five
or fewer steps, and the 440 direction offsets sit on a nearly flat
circular coordinate); with it in the model the sampler failed to mix
($\hat{R}$ up to 1.75 on the $\mu$ fixed effects, zero divergences). The
mean-direction effect is therefore estimated at population level (the
design holds the mean heading at the march direction, $0^\circ$), and
between-animal heterogeneity enters through the $\kappa$ random intercept.
Fit settings: 4 chains $\times$ (2000 tune + 1000 draws).


In [ ]:
# import ast
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import textwrap

# # ---------- load ----------
# RES_PATH = "kappa_only_model_sweep_results.parquet"
# res_df = pd.read_parquet(RES_PATH)

# # Backward compatibility
# if "pretty_name" not in res_df.columns and "name" in res_df.columns:
#     res_df = res_df.rename(columns={"name": "pretty_name"})

# # ---------- helpers ----------
# def _as_list(x):
#     """Coerce stored kappa_terms into a Python list."""
#     if x is None:
#         return []
#     if isinstance(x, (list, tuple)):
#         return list(x)
#     if isinstance(x, np.ndarray):
#         return x.ravel().tolist()
#     if isinstance(x, str):
#         # try parsing a stringified list first
#         try:
#             v = ast.literal_eval(x)
#             if isinstance(v, np.ndarray):
#                 return v.ravel().tolist()
#             if isinstance(v, (list, tuple)):
#                 return list(v)
#         except Exception:
#             pass
#         # fallback: treat string as a single token
#         return [x]
#     try:
#         if pd.isna(x):
#             return []
#     except Exception:
#         pass
#     return [str(x)]

# def _has_order_from_terms(terms):
#     terms = _as_list(terms)
#     return any(("t_order" in t) for t in terms if t != "1")

# def _has_order_from_name(name):
#     return isinstance(name, str) and ("order" in name.lower())

# # Determine grouping (prefer explicit kappa_terms)
# if "kappa_terms" in res_df.columns:
#     res_df["has_order"] = res_df["kappa_terms"].apply(_has_order_from_terms)
# else:
#     res_df["has_order"] = res_df["pretty_name"].apply(_has_order_from_name)

# # ---------- sort + ΔELPD ----------
# res_df = res_df.sort_values("elpd", ascending=False).reset_index(drop=True)
# best_elpd = float(res_df["elpd"].iloc[0])
# res_df["delta_elpd"] = res_df["elpd"] - best_elpd

# use_se = ("se" in res_df.columns) and (not res_df["se"].isna().all())

# # ---------- labels ----------
# def wrap_label(s, width=40):
#     return textwrap.fill(str(s), width=width)

# y = np.arange(len(res_df))
# y_labels = [wrap_label(n, width=40) for n in res_df["pretty_name"]]

# # ---------- split groups ----------
# g_incl = res_df[res_df["has_order"]].copy()
# g_excl = res_df[~res_df["has_order"]].copy()

# # keep y positions consistent with the global sorted table
# g_incl["y"] = g_incl.index.to_numpy()
# g_excl["y"] = g_excl.index.to_numpy()

# # ---------- plot ----------
# plt.figure(figsize=(4, 0.5 * len(res_df)), dpi=300)

# # Order included
# if use_se:
#     plt.errorbar(
#         g_incl["delta_elpd"], g_incl["y"],
#         xerr=g_incl["se"],
#         fmt="o", capsize=3, elinewidth=1, markersize=6,
#         label="Order included"
#     )
# else:
#     plt.plot(g_incl["delta_elpd"], g_incl["y"], "o", label="Order included")

# # Order excluded
# if use_se:
#     plt.errorbar(
#         g_excl["delta_elpd"], g_excl["y"],
#         xerr=g_excl["se"],
#         fmt="s", capsize=3, elinewidth=1, markersize=5,
#         label="Order excluded"
#     )
# else:
#     plt.plot(g_excl["delta_elpd"], g_excl["y"], "s", label="Order excluded")

# plt.axvline(0.0, linestyle="--", linewidth=1)
# plt.yticks(y, y_labels)
# plt.gca().invert_yaxis()
# plt.xlabel("ΔELPD vs best (PSIS-LOO)")
# plt.legend(loc="upper left", frameon=True)
# plt.tight_layout()
# plt.show()


In [ ]:
# from math import pi
# import numpy as np
# import pandas as pd
# import bambi as bmb
# import pymc as pm

# # Links & priors from unwrap (as in your example)
# from unwrap.prior import CircularUniformPrior
# from unwrap.link.bambi import circular_modulo, softplus
# from unwrap.link import inverse_softplus
# from pymc.distributions.transforms import LogExpM1 as Softplus

# # ----------------------------
# # Transform helpers (your defs)
# # ----------------------------
# def normalize(x):
#     return (x * 2.0) - 1.0

# def log2_scale(x, min_size=1., max_size=64.):
#     return np.log2(x / min_size) / np.log2(max_size / min_size)

# def inv_log2_scale(x, min_size=1., max_size=64.):
#     inv = x * np.log2(max_size / min_size)
#     inv = 2**inv
#     inv *= min_size
#     return inv

# def inv_normalize(x):
#     return (x - 1.0) / 2.0

# def exp_scale(x, min_size=1., max_size=64.):
#     return (x - min_size) / (max_size - min_size)

# # ----------------------------
# # Build modeling table
# # ----------------------------
# # >>> subsample per individual <<<

# n_samples_per_id = 20      # set the cap you want (or None to disable)
# rng_seed = 42              # for reproducibility
# if n_samples_per_id is not None:
#     df = (
#         merged_df
#         .groupby("animal_id", group_keys=False, observed=True)
#         .apply(
#             lambda g: g.sample(
#                 n=min(n_samples_per_id, len(g)),
#                 replace=False,
#                 random_state=rng_seed
#             ),
#             include_groups=True,
#         )
#         .reset_index()                 # animal_id comes back as an index -> column
#         .rename(columns={"index": "row_id"})  # optional, if you want to keep the old index
#     )
# else:
#     df = merged_df.copy()

# print(f"num_samples: {df.shape[0]}")

# # Ensure categorical id
# df["animal_id"] = df["animal_id"].astype("category")

# # Wrap y to (-pi, pi]
# df["y"] = ((df["direction"] + np.pi) % (2 * np.pi)) - np.pi

# # Choose scaling bounds for density (adjust to your experiment if needed)
# DENS_MIN, DENS_MAX = 1.0, 64.0  # <- change these to your actual density range

# # Transform predictors
# df["t_density"] = normalize(log2_scale(df["density"].astype(float), DENS_MIN, DENS_MAX))
# df["t_order"]   = normalize(df["order"].astype(float))

# # ----------------------------
# # Specify the hierarchical model
# # ----------------------------
# formula = bmb.Formula(
#     # "y ~ 1 + (1 | animal_id)",
#     "y ~ t_density * t_order + (1 | animal_id)",
#     "kappa ~ t_density * t_order + (1 | animal_id)"
# )


# priors = {
#     # μ parameters
#     "Intercept": bmb.Prior("VonMises", mu=0.0, kappa=3.0),
#     "t_density": bmb.Prior("Normal", mu=0.0, sigma=0.3),
#     "t_order":   bmb.Prior("Normal", mu=0.0, sigma=0.3),
#     "t_density:t_order": bmb.Prior("Normal", mu=0.0, sigma=0.2),
#     "1|animal_id": bmb.Prior(
#         "VonMises", mu=0.0,
#         kappa=bmb.Prior("Normal", mu=inverse_softplus(30.0), sigma=0.1, transform=Softplus())
#     ),
#     # κ parameters (pre-softplus)
#     "kappa": {
#         "Intercept": bmb.Prior("Normal", mu=inverse_softplus(2.0), sigma=0.5),
#         "t_density": bmb.Prior("Normal", mu=0.0, sigma=0.5),
#         "t_order":   bmb.Prior("Normal", mu=0.0, sigma=0.5),
#         "t_density:t_order": bmb.Prior("Normal", mu=0.0, sigma=0.25),
#         "1|animal_id": bmb.Prior(
#             "Normal", mu=0.0,
#             sigma=bmb.Prior("Normal", mu=inverse_softplus(1.0), sigma=0.25, transform=Softplus())
#         ),
#     },
# }

# model = bmb.Model(
#     formula=formula,
#     data=df,
#     family="vonmises",
#     priors=priors,
#     link={"mu": circular_modulo, "kappa": softplus},
#     center_predictors=True,
#     categorical="animal_id",
#     dropna=True,

# )
# model.build()
# # # Fit
# # # Increase draws/tune as needed; add target_accept=0.9 if you see divergences.
# print(model)
# idata_hierarchical = model.fit(
#     draws=1000, tune=1000, chains=4, cores=4,
#     target_accept=0.9, init="jitter+adapt_diag",
#     idata_kwargs=dict(
#         log_likelihood=True,
#         include_transformed=True,
#     ),
# )

In [ ]:
from math import pi
import numpy as np
import pandas as pd
import bambi as bmb
import pymc as pm

# For LOO
import arviz as az
import xarray as xr
from scipy.special import i0e

# Links & priors from unwrap (as in your example)
from unwrap.prior import CircularUniformPrior
from unwrap.link.bambi import circular_modulo, softplus
from unwrap.link import inverse_softplus
from pymc.distributions.transforms import LogExpM1 as Softplus

# ----------------------------
# Transform helpers (your defs)
# ----------------------------
def normalize(x):
    return (x * 2.0) - 1.0

def log2_scale(x, min_size=1.0, max_size=64.0):
    return np.log2(x / min_size) / np.log2(max_size / min_size)

def inv_log2_scale(x, min_size=1.0, max_size=64.0):
    inv = x * np.log2(max_size / min_size)
    inv = 2**inv
    inv *= min_size
    return inv

def inv_normalize(x):
    return (x - 1.0) / 2.0

def exp_scale(x, min_size=1.0, max_size=64.0):
    return (x - min_size) / (max_size - min_size)

# ----------------------------
# Build modeling table
# ----------------------------
n_samples_per_id = None
rng_seed = 123

if n_samples_per_id is not None:
    df = (
        merged_df
        .copy()
        .groupby("animal_id", group_keys=False, observed=True)
        .apply(lambda g: g.sample(
            n=min(n_samples_per_id, len(g)),
            replace=False,
            random_state=rng_seed
        ))
        .reset_index(drop=True)
    )
else:
    df = merged_df.copy()

print(f"num_samples: {df.shape[0]}")

# Ensure categorical id
df["animal_id"] = df["animal_id"].astype("category")

# Wrap y to (-pi, pi]
df["y"] = ((df["direction"] + np.pi) % (2 * np.pi)) - np.pi

# Choose scaling bounds for density (adjust to your experiment if needed)
DENS_MIN, DENS_MAX = 1.0, 64.0

# Transform predictors
df["t_density"] = normalize(log2_scale(df["density"].astype(float), DENS_MIN, DENS_MAX))
df["t_order"]   = normalize(df["order"].astype(float))

# ----------------------------
# Specify the hierarchical model
# ----------------------------
# Pooled mu component + kappa random intercept per animal.
# Each animal was observed under exactly ONE order-density condition,
# so order/density vary between animals only and no random slope is
# identifiable. Even the per-animal DIRECTION (mu) intercept offsets are
# weakly identifiable on this design (10% of animals have <= 5 steps,
# and the offsets lie on a flat circular coordinate): with them in the
# model the sampler failed to mix (rhat up to 1.75 on the mu fixed
# effects, ESS in the single digits, 0 divergences). The
# mean-direction effect is therefore estimated at population level (the
# design holds the mean heading at the march direction, 0 deg), and
# between-animal heterogeneity enters through the kappa random intercept.
formula = bmb.Formula(
    "y ~ t_density * t_order",
    "kappa ~ t_density * t_order + (1 | animal_id)",
)

priors = {
    # μ parameters
    "Intercept": bmb.Prior("VonMises", mu=0.0, kappa=3.0),
    "t_density": bmb.Prior("Normal", mu=0.0, sigma=0.5*np.pi),
    "t_order":   bmb.Prior("Normal", mu=0.0, sigma=0.5*np.pi),
    "t_density:t_order": bmb.Prior("Normal", mu=0.0, sigma=0.25*np.pi),

    # No mu random intercept: see the comment above the formula.
    # The between-animal direction offsets are weakly identifiable here
    # and including them broke sampling (see Sampling diagnostics).

    # κ parameters (pre-softplus)
    "kappa": {
        "Intercept": bmb.Prior("Normal", mu=inverse_softplus(2.0), sigma=0.5),
        "t_density": bmb.Prior("Normal", mu=0.0, sigma=0.6),
        "t_order":   bmb.Prior("Normal", mu=0.0, sigma=0.6),
        "t_density:t_order": bmb.Prior("Normal", mu=0.0, sigma=0.4),

        # κ random intercept
        "1|animal_id": bmb.Prior(
            "Normal", mu=0.0,
            sigma=bmb.Prior("Normal", mu=inverse_softplus(1.0), sigma=0.3, transform=Softplus())
        ),
    },
}

model = bmb.Model(
    formula=formula,
    data=df,
    family="vonmises",
    priors=priors,
    link={"mu": circular_modulo, "kappa": softplus},
    center_predictors=True,
    categorical="animal_id",
    dropna=True,
)
model.build()

print(model)

# ----------------------------
# Fit WITHOUT PyMC's automatic log_likelihood (it breaks w/ transformed RV names)
# ----------------------------
# Optional: seed numpy for any downstream randomness you might add
np.random.seed(rng_seed)

idata_hierarchical = model.fit(
    draws=1000, tune=2000, chains=4, cores=4,
    target_accept=0.9,
    init="jitter+adapt_diag",
    random_seed=rng_seed,
    idata_kwargs=dict(
        log_likelihood=False,
        include_transformed=True,
    ),
    omit_offsets=False,
)

# ----------------------------
# Add pointwise log-likelihood manually for elpd_loo
# ----------------------------
# Get posterior draws of response params at each observation
#
# N.B.: bambi 0.17.2 bug, tracked upstream at bambinos/bambi#1084:
# Model.predict(kind="response_params") applies the vonmises *mean* link,
# angle(exp(1j x)), to every component, so its "kappa" is the wrapped
# linear predictor (can be negative) instead of softplus(linear predictor).
# That breaks the log-likelihood below. Temporary fix until the official
# bambi fix ships: read the response parameters straight from the PyMC graph
# with compute_deterministics, which uses each component's actual link.

# Second bambi quirk needed for the call below: with random effects,
# Model.fit(omit_offsets=True) -- the default -- drops the non-centred
# "offset" free RVs (e.g. "1|animal_id_offset") from the posterior, and
# compute_deterministics reads every free RV by that exact name
# (KeyError: '1|animal_id_offset'). omit_offsets=False keeps them.
# Only relevant for random-effects models; plain GLMs are unaffected.
#
# Official (currently broken for the kappa component):
# preds = model.predict(idata_hierarchical, kind="response_params", inplace=False)
# mu    = preds.posterior["mu"]
# kappa = preds.posterior["kappa"]

import pymc as pm

params = pm.compute_deterministics(
    idata_hierarchical.posterior,
    var_names=["mu", "kappa"],
    model=model.backend.model,
    progressbar=False,
)
mu    = params["mu"]      # dims: (chain, draw, __obs__ or similar)
kappa = params["kappa"]   # dims: (chain, draw, __obs__ or similar)
assert (kappa.values > 0).all(), "kappa response params must be positive"
print("response params from PyMC graph OK (kappa all positive)")

# Find the observation dimension name used by bambi/preds
obs_dim = [d for d in mu.dims if d not in ("chain", "draw")][0]

# Observed y aligned to that obs dimension
y = xr.DataArray(
    df["y"].to_numpy(),
    dims=[obs_dim],
    coords={obs_dim: mu.coords[obs_dim]},
)

# von Mises logpdf: k*cos(y-mu) - log(2π) - log(I0(k))
# stable log(I0(k)) using i0e: I0(k) = i0e(k)*exp(k)
logI0 = np.log(i0e(kappa)) + kappa
loglik = kappa * np.cos(y - mu) - np.log(2 * np.pi) - logI0

# Attach to idata in the ArviZ-standard location
idata_hierarchical.add_groups({"log_likelihood": xr.Dataset({"y": loglik})})

# ----------------------------
# Example: LOO / elpd_loo
# ----------------------------
loo = az.loo(idata_hierarchical, var_name="y")
print(loo)


num_samples: 18641


       Formula: y ~ t_density * t_order
                kappa ~ t_density * t_order + (1 | animal_id)
        Family: vonmises
          Link: mu = circular_modulo
                kappa = softplus
  Observations: 18641
        Priors: 
    target = mu
        Common-level effects
            Intercept ~ VonMises(mu: 0.0, kappa: 3.0)
            t_density ~ Normal(mu: 0.0, sigma: 1.5708)
            t_order ~ Normal(mu: 0.0, sigma: 1.5708)
            t_density:t_order ~ Normal(mu: 0.0, sigma: 0.7854)
    target = kappa
        Common-level effects
            kappa_Intercept ~ Normal(mu: 1.8546, sigma: 0.5)
            kappa_t_density ~ Normal(mu: 0.0, sigma: 0.6)
            kappa_t_order ~ Normal(mu: 0.0, sigma: 0.6)
            kappa_t_density:t_order ~ Normal(mu: 0.0, sigma: 0.4)
        
        Group-level effects
            kappa_1|animal_id ~ Normal(mu: 0.0, sigma: Normal(mu: 0.5413, sigma: 0.3, transform:
                log_exp_m1))



Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [Intercept, t_density, t_order, t_density:t_order, kappa_Intercept, kappa_t_density, kappa_t_order, kappa_t_density:t_order, kappa_1|animal_id_sigma_log_exp, kappa_1|animal_id_offset]


In [ ]:
loo = az.loo(idata_hierarchical, var_name="y")
p_eff = float(loo.p_loo)
N = 18641
print("p_loo (effective params):", p_eff)
print("N / p_loo:", N / p_eff)


/home/jake/.venvs/unwrap-env/lib/python3.12/site-packages/arviz/stats/stats.py:782: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(


p_loo (effective params): 224.25244767619733
N / p_loo: 83.12506816833553


In [ ]:
print("fraction k>0.7:", np.mean(loo.pareto_k.values > 0.7))

fraction k>0.7: 0.001931226865511507


In [ ]:
import arviz as az
summary_df = az.summary(idata_hierarchical, circ_var_names=["Intercept"])

In [ ]:
summary_df

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
Intercept,-0.067,0.014,-0.090,-0.039,0.000,0.000,3795.0,3228.0,1.00
t_density,-0.144,0.020,-0.181,-0.107,0.000,0.000,3974.0,3245.0,1.00
t_order,0.073,0.017,0.042,0.106,0.000,0.000,3790.0,3579.0,1.00
t_density:t_order,0.156,0.028,0.101,0.207,0.000,0.000,3825.0,3345.0,1.00
kappa_Intercept,0.428,0.111,0.228,0.637,0.004,0.002,682.0,1288.0,1.01
...,...,...,...,...,...,...,...,...,...
kappa_1|animal_id[435],-2.019,1.186,-4.262,0.090,0.016,0.021,6544.0,3242.0,1.00
kappa_1|animal_id[436],-0.231,1.473,-2.990,2.278,0.022,0.028,5763.0,2714.0,1.00
kappa_1|animal_id[437],2.078,0.956,0.357,4.004,0.012,0.020,7171.0,2603.0,1.00
kappa_1|animal_id[438],-0.468,0.698,-1.793,0.793,0.013,0.017,3171.0,1971.0,1.00


In [ ]:
summary_df[(summary_df["ess_bulk"]<400) | (summary_df["ess_tail"]<400)]

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat


In [ ]:
summary_df[(summary_df["r_hat"]>1.01)]

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat


## Sampling diagnostics

ESS / $\hat{R}$ tables for the final fit. With the pooled-$\mu$
specification, **all reported parameters pass**: $\hat{R} \le 1.01$ and
ESS well above 400 for the fixed effects (Intercept, `t_density`,
`t_order`, the interaction, and the corresponding `kappa_*` terms) used
for the scientific claims, and for the $\kappa$ random intercept
(aggregate and hyperparameter). The reason the direction component is not
given random effects is that the 440 between-animal direction offsets are
weakly identified under this design (one condition per animal; 10% of
animals have five or fewer steps): including them produced $\hat{R}$ up to
1.75 with zero divergences -- a weakly identified hierarchy, not a
sampling failure -- so they are estimated at population level instead.


In [ ]:
# az.plot_trace(idata_hierarchical, var_names=["Intercept", "t_order", "t_density", "t_density:t_order"])
# plt.tight_layout();

In [ ]:
# az.plot_trace(idata_hierarchical, var_names=["1|animal_id_kappa", "kappa_Intercept", "kappa_t_order", "kappa_t_density", "kappa_t_density:t_order"])
# plt.tight_layout();

In [ ]:
# az.plot_posterior(idata_hierarchical, var_names=["1|animal_id_kappa", "kappa_Intercept", "kappa_t_order", "kappa_t_density", "kappa_t_density:t_order", "kappa_1|animal_id_sigma"]);
# az.plot_posterior(idata_hierarchical, var_names=["Intercept", "1|animal_id"], circular=True);

In [ ]:
# az.plot_posterior(idata_hierarchical, var_names=["kappa_1|animal_id"]);


In [ ]:
# # --- Predict directional alignment = A(kappa) * cos(mu) and replicate two panels ---
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import arviz as az
# from scipy.special import i0, i1  # A(kappa) = I1/I0

# # same transforms you used in the model
# from math import pi
# def normalize(x): return x * 2.0 - 1.0
# def log2_scale(x, min_size=1., max_size=64.):
#     return np.log2(x / min_size) / np.log2(max_size / min_size)

# DENS_MIN, DENS_MAX = 1.0, 64.0  # match training bounds

# def softplus(x):
#     # numerically stable softplus
#     return np.log1p(np.exp(x))

# def circular_modulo(x):
#     # wrap to (-pi, pi]
#     return (x + np.pi) % (2 * np.pi) - np.pi

# def A_of_kappa(kappa):
#     return i1(kappa) / i0(kappa)

# def posterior_linpred(idata, prefix, X_df):
#     """
#     Build linear predictor from posterior samples for parameter set `prefix`.
#     prefix == "" for mu terms (e.g., 'Intercept', 't_density', ...).
#     prefix == "kappa_" for kappa terms (e.g., 'kappa_Intercept', ...).
#     Returns array shape (S, n_rows).
#     """
#     post = idata.posterior
#     # collect relevant coef names
#     if prefix:
#         coefs = [v for v in post.data_vars if v.startswith(prefix)]
#         key_intercept = f"{prefix}Intercept"
#     else:
#         # mu coefs are the base names (no prefix)
#         coefs = [v for v in post.data_vars if (not v.startswith("kappa_"))]
#         key_intercept = "Intercept"

#     if key_intercept not in post:
#         raise ValueError(f"Intercept for prefix '{prefix}' not found in posterior.")

#     def flat(name):
#         return post[name].stack(sample=("chain", "draw")).values  # (S,)
#     S = post.sizes["chain"] * post.sizes["draw"]

#     n = len(X_df)
#     eta = np.tile(flat(key_intercept)[:, None], (1, n))  # (S, n)

#     for coef in coefs:
#         if coef == key_intercept:
#             continue
#         col = coef.replace(prefix, "")  # strip prefix for kappa
#         if col in X_df.columns:
#             eta += flat(coef)[:, None] * X_df[col].to_numpy()[None, :]

#     return eta  # (S, n)

# # Build design matrices for both μ and κ submodels you fit:
# # μ: y ~ t_density + t_order + t_density:t_order
# # κ: kappa ~ t_density + t_order   (add interaction here if you included it)
# def make_X(order_vals, density_vals, idata):
#     X = pd.DataFrame({"order": order_vals, "density": density_vals})
#     X["t_order"]   = normalize(X["order"].astype(float))
#     X["t_density"] = normalize(log2_scale(X["density"].astype(float), DENS_MIN, DENS_MAX))
#     # include interaction column if present in posterior for either submodel
#     if ("t_density:t_order" in idata.posterior.data_vars or
#         "kappa_t_density:t_order" in idata.posterior.data_vars):
#         X["t_density:t_order"] = X["t_density"] * X["t_order"]
#     return X

# def posterior_alignment(idata, X):
#     """
#     Returns dict of posterior summaries for alignment = A(kappa) * cos(mu).
#     """
#     # Linear predictors
#     eta_mu    = posterior_linpred(idata, prefix="",       X_df=X)  # (S, n)
#     eta_kappa = posterior_linpred(idata, prefix="kappa_", X_df=X)  # (S, n)

#     # Links
#     mu    = circular_modulo(eta_mu)     # (-pi, pi]
#     kappa = softplus(eta_kappa)         # > 0

#     # Alignment = mean resultant length * cos(mu - h); here h = 0
#     A = A_of_kappa(kappa)               # (S, n)
#     align = A * np.cos(mu)              # (S, n)

#     mean = align.mean(axis=0)
#     hdi68 = az.hdi(align, hdi_prob=0.68, skipna=True)  # shape (n, 2)
#     return {"mean": mean, "low": hdi68[:, 0], "high": hdi68[:, 1]}

# # ---- prediction grids ----
# dens_levels  = np.sort(merged_df["density"].unique())
# order_levels = np.sort(merged_df["order"].unique())
# order_grid   = np.linspace(order_levels.min(), order_levels.max(), 50)
# density_grid = np.geomspace(max(dens_levels.min(), 1e-6), dens_levels.max(), 50)

# # Panel (i): vs Order, colored by Density
# pred_i = {}
# for d in dens_levels:
#     X = make_X(order_grid, np.full_like(order_grid, d, dtype=float), idata_hierarchical)
#     # keep only cols that appear in posterior names
#     keep_cols = ["t_order", "t_density"] + ([ "t_density:t_order" ] if "t_density:t_order" in X.columns else [])
#     pred_i[d] = posterior_alignment(idata_hierarchical, X[keep_cols])

# # Panel (ii): vs Density, colored by Order
# pred_ii = {}
# for o in order_levels:
#     X = make_X(np.full_like(density_grid, o, dtype=float), density_grid, idata_hierarchical)
#     keep_cols = ["t_order", "t_density"] + ([ "t_density:t_order" ] if "t_density:t_order" in X.columns else [])
#     pred_ii[o] = posterior_alignment(idata_hierarchical, X[keep_cols])

# # ---- plot with legends (same style) ----
# plt.rcParams.update({
#     "font.size": 8, "ytick.major.width": 1, "xtick.major.width": 1,
#     "axes.linewidth": 1, "font.family": "Arial"
# })

# fig, axs = plt.subplots(1, 2, figsize=(6.2, 2.2), dpi=300)
# ax1, ax2 = axs

# cmap_density = plt.cm.Reds     # for density series (panel i)
# cmap_order   = plt.cm.Purples  # for order series (panel ii)

# # Panel i: Alignment vs Order (colored by Density)
# colors_d = np.linspace(0.3, 0.95, len(dens_levels))
# h_i, l_i = [], []
# for i, d in enumerate(dens_levels):
#     col = cmap_density(colors_d[i])
#     y = pred_i[d]["mean"]; lo = pred_i[d]["low"]; hi = pred_i[d]["high"]
#     line, = ax1.plot(order_grid, y, color=col, lw=1.5)
#     ax1.fill_between(order_grid, lo, hi, color=col, alpha=0.25, linewidth=0)
#     h_i.append(line); l_i.append(f"{d:g}")
# ax1.set_xlabel("Order"); ax1.set_ylabel("Directional alignment")
# ax1.set_xlim(order_grid.min(), order_grid.max())
# ax1.set_yticks([-0.5, 0.0, 0.5, 1.0]); ax1.set_ylim(-0.05, 1.05) 
# ax1.legend(h_i, l_i, title="Density", loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)

# # Panel ii: Alignment vs Density (colored by Order)
# colors_o = np.linspace(0.3, 0.95, len(order_levels))
# h_ii, l_ii = [], []
# for i, o in enumerate(order_levels):
#     col = cmap_order(colors_o[i])
#     y = pred_ii[o]["mean"]; lo = pred_ii[o]["low"]; hi = pred_ii[o]["high"]
#     line, = ax2.plot(density_grid, y, color=col, lw=1.5)
#     ax2.fill_between(density_grid, lo, hi, color=col, alpha=0.25, linewidth=0)
#     h_ii.append(line); l_ii.append(f"{o:.1f}" if (o % 1) else f"{int(o)}")
# ax2.set_xscale("log", base=2)
# ax2.set_xlabel("Density"); ax2.set_xlim(density_grid.min(), density_grid.max())
# ax2.set_xticks([1, 2, 4, 8, 16, 32, 64])
# ax2.set_xticklabels([1, 2, 4, 8, 16, 32, 64])
# ax2.set_yticks([-0.5, 0.0, 0.5, 1.0]); ax2.set_ylim(-0.05, 1.05)
# ax2.legend(h_ii, l_ii, title="Order", loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)

# fig.tight_layout()
# plt.show()


## Posterior predictive checks

The final set of figures validates the hierarchical fit with the alignment
statistic and the predictive density at the march direction:

- **Alignment surfaces**: the posterior predictive alignment
  $\mathcal{A}(\mathbf{x})$ as a function of order (one panel per density)
  and of density (one panel per order), with open markers at the observed
  cells from a hierarchical bootstrap of the data (resample animals, then
  steps within animals). Alignment rises with order and levels off once order
  is high; the change across density 1–64 m$^{-2}$ is comparatively small.
- **Predictive density at $0^\circ$**: the posterior density mass the model
  places exactly at the march direction, over order and over density, with a
  dashed uniform reference ($1/2\pi$).
- **Coefficient posteriors**: the fixed-effect posteriors for
  $\mu(\mathbf{x})$ and $\kappa(\mathbf{x})$ (pre-link scale) with 95 %
  HDIs, next to the $\Delta$ELPD comparison.
- **Cell-level PPC** (dots + errorbars at each observed order–density cell):
  open markers are the empirical alignment from the data bootstrap; filled
  markers are the posterior predictive alignment, obtained by simulating the
  same number of steps per animal from the individual-level likelihood
  (fixed effects + that animal's random effects) and applying the identical
  bootstrap to the simulated data.
- **Individual-level PPC** (36 panels, one per order–density condition): the
  per-animal alignment statistic, data (within-individual bootstrap of steps)
  vs model (posterior predictive using each animal's own random effects);
  dashed lines mark zero alignment.

Where the data and model markers and intervals agree across all 36 cells, the
hierarchical von Mises GLM reproduces the main features of the directional
alignment data at both the population and the individual level.


In [ ]:
# ============================================================
# Plot 3 — Predicted alignment = A(kappa) * cos(mu)
# Two panels; width matches the 2-subplot figure, height set to look nice.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import arviz as az
from scipy.special import i0, i1

# ----------------------------
# Style
# ----------------------------
DPI = 300
AX_LW     = 0.8
TICK_LW   = 0.8
TICK_LEN  = 2.5
LINE_LW   = 1.0
FONT_SIZE = 7

mpl.rcParams.update({
    "font.size": FONT_SIZE,
    "axes.labelsize": FONT_SIZE,
    "axes.titlesize": FONT_SIZE,
    "legend.fontsize": FONT_SIZE,
    "xtick.labelsize": FONT_SIZE,
    "ytick.labelsize": FONT_SIZE,

    "axes.linewidth": AX_LW,
    "lines.linewidth": LINE_LW,

    "xtick.major.width": TICK_LW,
    "ytick.major.width": TICK_LW,
    "xtick.major.size": TICK_LEN,
    "ytick.major.size": TICK_LEN,

    "savefig.dpi": DPI,
    "figure.dpi": DPI,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

def _style_axis(ax):
    for s in ax.spines.values():
        s.set_linewidth(AX_LW)

# ----------------------------
# transforms
# ----------------------------
def normalize(x): 
    return x * 2.0 - 1.0

def log2_scale(x, min_size=1., max_size=64.):
    return np.log2(x / min_size) / np.log2(max_size / min_size)

DENS_MIN, DENS_MAX = 1.0, 64.0

def softplus(x):
    x = np.asarray(x)
    return np.where(x > 20, x, np.log1p(np.exp(x)))

def circular_modulo(x):
    return (x + np.pi) % (2 * np.pi) - np.pi

def A_of_kappa(kappa):
    return i1(kappa) / i0(kappa)

def posterior_linpred(idata, prefix, X_df):
    post = idata.posterior

    if prefix:
        coefs = [v for v in post.data_vars if v.startswith(prefix)]
        key_intercept = f"{prefix}Intercept"
    else:
        coefs = [v for v in post.data_vars if (not v.startswith("kappa_"))]
        key_intercept = "Intercept"

    if key_intercept not in post:
        raise ValueError(f"Intercept for prefix '{prefix}' not found in posterior.")

    def flat(name):
        return post[name].stack(sample=("chain", "draw")).values

    n = len(X_df)
    eta = flat(key_intercept)[:, None] * np.ones((1, n), dtype=float)

    for coef in coefs:
        if coef == key_intercept:
            continue
        col = coef.replace(prefix, "")
        if col in X_df.columns:
            eta += flat(coef)[:, None] * X_df[col].to_numpy()[None, :]

    return eta

def _compute_training_means(merged_df):
    t_order = normalize(merged_df["order"].astype(float).to_numpy())
    t_density = normalize(
        log2_scale(merged_df["density"].astype(float).to_numpy(), DENS_MIN, DENS_MAX)
    )
    return {"t_order": np.mean(t_order), "t_density": np.mean(t_density)}

TRAIN_MEANS = _compute_training_means(merged_df)

def make_X(order_vals, density_vals, idata, center_predictors=True, interaction_from_centered=True):
    X = pd.DataFrame({"order": order_vals, "density": density_vals})
    X["t_order"]   = normalize(X["order"].astype(float))
    X["t_density"] = normalize(log2_scale(X["density"].astype(float), DENS_MIN, DENS_MAX))

    if center_predictors:
        X["t_order"]   = X["t_order"]   - TRAIN_MEANS["t_order"]
        X["t_density"] = X["t_density"] - TRAIN_MEANS["t_density"]

    has_interaction = (
        ("t_density:t_order" in idata.posterior.data_vars) or
        ("kappa_t_density:t_order" in idata.posterior.data_vars)
    )
    if has_interaction:
        if interaction_from_centered:
            X["t_density:t_order"] = X["t_density"] * X["t_order"]
        else:
            X_raw_order   = normalize(pd.Series(order_vals).astype(float))
            X_raw_density = normalize(
                log2_scale(pd.Series(density_vals).astype(float), DENS_MIN, DENS_MAX)
            )
            X["t_density:t_order"] = np.asarray(X_raw_density) * np.asarray(X_raw_order)

    return X

def posterior_alignment(idata, X):
    eta_mu    = posterior_linpred(idata, prefix="",       X_df=X)
    eta_kappa = posterior_linpred(idata, prefix="kappa_", X_df=X)

    mu    = circular_modulo(eta_mu)
    kappa = softplus(eta_kappa)

    A = A_of_kappa(kappa)
    align = A * np.cos(mu)

    mean = align.mean(axis=0)
    hdi = az.hdi(align, hdi_prob=0.9, skipna=True)
    return {"mean": mean, "low": hdi[:, 0], "high": hdi[:, 1]}

# -----------------------------
# prediction grids
# -----------------------------
# DESCENDING so legends list largest -> smallest
dens_levels  = np.sort(merged_df["density"].unique())[::-1]
order_levels = np.sort(merged_df["order"].unique())[::-1]

order_grid   = np.linspace(order_levels.min(), order_levels.max(), 50)
density_grid = np.geomspace(max(dens_levels.min(), 1e-6), dens_levels.max(), 50)

pred_i = {}
for d in dens_levels:
    X = make_X(
        order_grid,
        np.full_like(order_grid, d, dtype=float),
        idata_hierarchical,
        center_predictors=True,
        interaction_from_centered=True,
    )
    keep_cols = ["t_order", "t_density"] + (["t_density:t_order"] if "t_density:t_order" in X.columns else [])
    pred_i[d] = posterior_alignment(idata_hierarchical, X[keep_cols])

pred_ii = {}
for o in order_levels:
    X = make_X(
        np.full_like(density_grid, o, dtype=float),
        density_grid,
        idata_hierarchical,
        center_predictors=True,
        interaction_from_centered=True,
    )
    keep_cols = ["t_order", "t_density"] + (["t_density:t_order"] if "t_density:t_order" in X.columns else [])
    pred_ii[o] = posterior_alignment(idata_hierarchical, X[keep_cols])

# -----------------------------
# plot sizing
# -----------------------------
FIG_W = 1.8 * 2 + 0.6
FIG_H = 1.55

# Figure: the matplotlib 'how' lives in locust_density_plots.py.
plots.plot_alignment_prediction_centered(
    order_grid, density_grid, dens_levels, order_levels, pred_i, pred_ii
)


/tmp/ipykernel_3881822/1168597650.py:143: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(align, hdi_prob=0.9, skipna=True)
/tmp/ipykernel_3881822/1168597650.py:143: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(align, hdi_prob=0.9, skipna=True)
/tmp/ipykernel_3881822/1168597650.py:143: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(align, hdi_prob=0.9, skipna=True)
/tmp/ipykernel_3881822/1168597650.py:143: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(align, hdi_prob=0.9, skipna=True)


/tmp/ipykernel_3881822/1168597650.py:143: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(align, hdi_prob=0.9, skipna=True)
/tmp/ipykernel_3881822/1168597650.py:143: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(align, hdi_prob=0.9, skipna=True)
/tmp/ipykernel_3881822/1168597650.py:143: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(align, hdi_prob=0.9, skipna=True)
/tmp/ipykernel_3881822/1168597650.py:143: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(align, hdi_prob=0.9, skipna=True)


/tmp/ipykernel_3881822/1168597650.py:143: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(align, hdi_prob=0.9, skipna=True)
/tmp/ipykernel_3881822/1168597650.py:143: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(align, hdi_prob=0.9, skipna=True)
/tmp/ipykernel_3881822/1168597650.py:143: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(align, hdi_prob=0.9, skipna=True)
/tmp/ipykernel_3881822/1168597650.py:143: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(align, hdi_prob=0.9, skipna=True)


Saved figures to: figures/locust/order_density_alignment.pdf; figures/locust/order_density_alignment.png


In [ ]:
# ============================================================
# Plot 3 (ELPD@0) — Predictive density at y=0
# Two panels (Order | Density)
#
# CHANGE REQUESTED:
#   - compute posterior mean + HDI in *log space*, then exp() for plotting
#   - y-axis label: "Predictive density"
#   - y-lims: 0 to 1
#   - dashed uniform reference at density = 1/(2*pi)
#   - keep Arial + transparent backgrounds
# ============================================================

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import arviz as az
from scipy.stats import vonmises

# ----------------------------
# Style
# ----------------------------
DPI = 300
AX_LW     = 0.8
TICK_LW   = 0.8
TICK_LEN  = 2.5
LINE_LW   = 1.0
FONT_SIZE = 7

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": FONT_SIZE,
    "axes.labelsize": FONT_SIZE,
    "axes.titlesize": FONT_SIZE,
    "legend.fontsize": FONT_SIZE,
    "xtick.labelsize": FONT_SIZE,
    "ytick.labelsize": FONT_SIZE,

    "axes.linewidth": AX_LW,
    "lines.linewidth": LINE_LW,

    "xtick.major.width": TICK_LW,
    "ytick.major.width": TICK_LW,
    "xtick.major.size": TICK_LEN,
    "ytick.major.size": TICK_LEN,

    "savefig.dpi": DPI,
    "figure.dpi": DPI,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

def _style_axis(ax):
    for s in ax.spines.values():
        s.set_linewidth(AX_LW)

# ----------------------------
# transforms / helpers
# ----------------------------
def normalize(x):
    return x * 2.0 - 1.0

def log2_scale(x, min_size=1., max_size=64.):
    return np.log2(x / min_size) / np.log2(max_size / min_size)

DENS_MIN, DENS_MAX = 1.0, 64.0

def softplus(x):
    x = np.asarray(x)
    return np.where(x > 20, x, np.log1p(np.exp(x)))

def circular_modulo(x):
    return (x + np.pi) % (2 * np.pi) - np.pi

def posterior_linpred(idata, prefix, X_df):
    post = idata.posterior

    if prefix:
        coefs = [v for v in post.data_vars if v.startswith(prefix)]
        key_intercept = f"{prefix}Intercept"
    else:
        coefs = [v for v in post.data_vars if (not v.startswith("kappa_"))]
        key_intercept = "Intercept"

    if key_intercept not in post:
        raise ValueError(f"Intercept for prefix '{prefix}' not found in posterior.")

    def flat(name):
        return post[name].stack(sample=("chain", "draw")).values

    n = len(X_df)
    eta = flat(key_intercept)[:, None] * np.ones((1, n), dtype=float)

    for coef in coefs:
        if coef == key_intercept:
            continue
        col = coef.replace(prefix, "")
        if col in X_df.columns:
            eta += flat(coef)[:, None] * X_df[col].to_numpy()[None, :]

    return eta

def _compute_training_means(merged_df):
    t_order = normalize(merged_df["order"].astype(float).to_numpy())
    t_density = normalize(
        log2_scale(merged_df["density"].astype(float).to_numpy(), DENS_MIN, DENS_MAX)
    )
    return {"t_order": np.mean(t_order), "t_density": np.mean(t_density)}

TRAIN_MEANS = _compute_training_means(merged_df)

def make_X(order_vals, density_vals, idata, center_predictors=True, interaction_from_centered=True):
    X = pd.DataFrame({"order": order_vals, "density": density_vals})
    X["t_order"]   = normalize(X["order"].astype(float))
    X["t_density"] = normalize(log2_scale(X["density"].astype(float), DENS_MIN, DENS_MAX))

    if center_predictors:
        X["t_order"]   = X["t_order"]   - TRAIN_MEANS["t_order"]
        X["t_density"] = X["t_density"] - TRAIN_MEANS["t_density"]

    has_interaction = (
        ("t_density:t_order" in idata.posterior.data_vars) or
        ("kappa_t_density:t_order" in idata.posterior.data_vars)
    )
    if has_interaction:
        if interaction_from_centered:
            X["t_density:t_order"] = X["t_density"] * X["t_order"]
        else:
            X_raw_order   = normalize(pd.Series(order_vals).astype(float))
            X_raw_density = normalize(
                log2_scale(pd.Series(density_vals).astype(float), DENS_MIN, DENS_MAX)
            )
            X["t_density:t_order"] = np.asarray(X_raw_density) * np.asarray(X_raw_order)

    return X

# ----------------------------
# posterior predictive density at y=0 (compute in log, plot in exp(log))
# ----------------------------
def posterior_elpd0(idata, X, hdi_prob=0.9):
    eta_mu    = posterior_linpred(idata, prefix="",       X_df=X)
    eta_kappa = posterior_linpred(idata, prefix="kappa_", X_df=X)

    mu    = circular_modulo(eta_mu)
    kappa = softplus(eta_kappa)

    # log predictive density at y=0 for each posterior draw
    logp0 = vonmises.logpdf(0.0, kappa, loc=mu)  # (S, n_points)

    # mean and HDI in log space
    mean_log = np.mean(logp0, axis=0)
    hdi_log  = az.hdi(logp0, hdi_prob=hdi_prob, skipna=True)

    # exponentiate for plotting
    return {
        "mean": np.exp(mean_log),
        "low":  np.exp(hdi_log[:, 0]),
        "high": np.exp(hdi_log[:, 1]),
        "mean_log": mean_log,     # kept in case you want it later
        "low_log":  hdi_log[:, 0],
        "high_log": hdi_log[:, 1],
    }

# -----------------------------
# prediction grids
# -----------------------------
dens_levels  = np.sort(merged_df["density"].unique())[::-1]
order_levels = np.sort(merged_df["order"].unique())[::-1]

order_grid   = np.linspace(order_levels.min(), order_levels.max(), 50)
density_grid = np.geomspace(max(dens_levels.min(), 1e-6), dens_levels.max(), 50)

pred_i = {}
for d in dens_levels:
    X = make_X(
        order_grid,
        np.full_like(order_grid, d, dtype=float),
        idata_hierarchical,
        center_predictors=True,
        interaction_from_centered=True,
    )
    keep_cols = ["t_order", "t_density"] + (["t_density:t_order"] if "t_density:t_order" in X.columns else [])
    pred_i[d] = posterior_elpd0(idata_hierarchical, X[keep_cols], hdi_prob=0.9)

pred_ii = {}
for o in order_levels:
    X = make_X(
        np.full_like(density_grid, o, dtype=float),
        density_grid,
        idata_hierarchical,
        center_predictors=True,
        interaction_from_centered=True,
    )
    keep_cols = ["t_order", "t_density"] + (["t_density:t_order"] if "t_density:t_order" in X.columns else [])
    pred_ii[o] = posterior_elpd0(idata_hierarchical, X[keep_cols], hdi_prob=0.9)

# -----------------------------
# plot sizing
# -----------------------------
FIG_W = 1.8 * 2 + 0.6
FIG_H = 1.55

# Figure: the matplotlib 'how' lives in locust_density_plots.py.
plots.plot_elpd0_prediction_centered(
    order_grid, density_grid, dens_levels, order_levels, pred_i, pred_ii
)


/tmp/ipykernel_3881822/1409058159.py:152: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi_log  = az.hdi(logp0, hdi_prob=hdi_prob, skipna=True)
/tmp/ipykernel_3881822/1409058159.py:152: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi_log  = az.hdi(logp0, hdi_prob=hdi_prob, skipna=True)
/tmp/ipykernel_3881822/1409058159.py:152: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi_log  = az.hdi(logp0, hdi_prob=hdi_prob, skipna=True)
/tmp/ipykernel_3881822/1409058159.py:152: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi_log  = az.hdi(log

/tmp/ipykernel_3881822/1409058159.py:152: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi_log  = az.hdi(logp0, hdi_prob=hdi_prob, skipna=True)
/tmp/ipykernel_3881822/1409058159.py:152: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi_log  = az.hdi(logp0, hdi_prob=hdi_prob, skipna=True)
/tmp/ipykernel_3881822/1409058159.py:152: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi_log  = az.hdi(logp0, hdi_prob=hdi_prob, skipna=True)
/tmp/ipykernel_3881822/1409058159.py:152: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi_log  = az.hdi(log

/tmp/ipykernel_3881822/1409058159.py:152: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi_log  = az.hdi(logp0, hdi_prob=hdi_prob, skipna=True)
/tmp/ipykernel_3881822/1409058159.py:152: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi_log  = az.hdi(logp0, hdi_prob=hdi_prob, skipna=True)


Saved figures to: figures/locust/order_density_elpd0.pdf; figures/locust/order_density_elpd0.png


In [ ]:
"""
2 rows of POLAR small-multiples with *multiple* von Mises predictive PDFs per panel.

Row 1 (columns = each ORDER level):
    within each panel: sweep DENSITY levels (colored by Reds, like your ELPD plot)

Row 2 (columns = each DENSITY level):
    within each panel: sweep ORDER levels (colored by Purples, like your ELPD plot)

Each curve shows:
  - median posterior predictive PDF (line)
  - 90% HDI band for the PDF (fill)   <-- now clearly visible
  - “HDI bars” as arcs on the inner ring: 90% interval for posterior μ (circular)

ASSUMES you already have from your ELPD code:
  - merged_df
  - idata_hierarchical
  - make_X(...)
  - posterior_linpred(...)
  - softplus(...)
  - circular_modulo(...)
"""

import numpy as np
import arviz as az
def vonmises_pdf(theta, mu, kappa):
    return np.exp(kappa * np.cos(theta - mu)) / (2 * np.pi * np.i0(kappa))

def angle_wrap(x):
    return (x + np.pi) % (2*np.pi) - np.pi

def circular_mean(mu_draws):
    return np.angle(np.mean(np.exp(1j * mu_draws)))

def mu_interval_central(mu_draws, prob=0.9):
    """Practical circular 'HDI-like' interval: unwrap around circular mean then take central interval."""
    mu0 = circular_mean(mu_draws)
    d   = np.angle(np.exp(1j * (mu_draws - mu0)))  # deltas in (-pi, pi]
    a = (1 - prob) / 2 * 100
    b = (1 + prob) / 2 * 100
    lo_d, hi_d = np.percentile(d, [a, b])
    lo = angle_wrap(mu0 + lo_d)
    hi = angle_wrap(mu0 + hi_d)
    return mu0, lo, hi

def _subsample_draws(mu, kappa, max_draws, rng):
    S = mu.shape[0]
    if S <= max_draws:
        return mu, kappa
    idx = rng.choice(S, size=max_draws, replace=False)
    return mu[idx], kappa[idx]

def posterior_mu_kappa_at(idata, order_val, density_val):
    X = make_X(
        np.array([order_val], dtype=float),
        np.array([density_val], dtype=float),
        idata,
        center_predictors=True,
        interaction_from_centered=True,
    )
    keep_cols = ["t_order", "t_density"] + (["t_density:t_order"] if "t_density:t_order" in X.columns else [])
    X = X[keep_cols]

    eta_mu    = posterior_linpred(idata, prefix="",       X_df=X)      # (S, 1)
    eta_kappa = posterior_linpred(idata, prefix="kappa_", X_df=X)      # (S, 1)

    mu    = circular_modulo(eta_mu[:, 0])
    kappa = softplus(eta_kappa[:, 0])
    return mu, kappa

def posterior_pdf_summary(idata, order_val, density_val, theta_grid, hdi_prob=0.9, max_draws=1200, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)

    mu, kappa = posterior_mu_kappa_at(idata, order_val, density_val)
    mu, kappa = _subsample_draws(mu, kappa, max_draws, rng)

    pdfs = vonmises_pdf(theta_grid[None, :], mu[:, None], kappa[:, None])  # (S, T)

    med = np.median(pdfs, axis=0)
    hdi = az.hdi(pdfs, hdi_prob=hdi_prob, skipna=True)  # (T, 2)
    lo, hi = hdi[:, 0], hdi[:, 1]

    mu0, mu_lo, mu_hi = mu_interval_central(mu, prob=hdi_prob)
    return dict(med=med, lo=lo, hi=hi, mu0=mu0, mu_lo=mu_lo, mu_hi=mu_hi)
# Levels (your actual unique levels, descending for the legend order)
dens_levels  = np.sort(merged_df["density"].unique())[::-1]
order_levels = np.sort(merged_df["order"].unique())[::-1]

# Angle grid + precomputed PDF summaries for every (order, density) cell
HDI_PROB  = 0.90
MAX_DRAWS = 1200      # subsample posterior draws for speed

rng   = np.random.default_rng(123)
theta = np.linspace(-np.pi, np.pi, 900, endpoint=False)


summ = {}
peak_max = 0.0
for o in order_levels:
    for d in dens_levels:
        res = posterior_pdf_summary(
            idata_hierarchical, float(o), float(d),
            theta_grid=theta,
            hdi_prob=HDI_PROB,
            max_draws=MAX_DRAWS,
            rng=rng
        )
        summ[(float(o), float(d))] = res
        peak_max = max(peak_max, res["hi"].max())


# Figure: polar small-multiples of predictive PDFs, drawn by the sibling script.
plots.plot_polar_pdfs_grid(theta, summ, order_levels, dens_levels, peak_max)


/tmp/ipykernel_3881822/652050621.py:81: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(pdfs, hdi_prob=hdi_prob, skipna=True)  # (T, 2)
/tmp/ipykernel_3881822/652050621.py:81: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(pdfs, hdi_prob=hdi_prob, skipna=True)  # (T, 2)
/tmp/ipykernel_3881822/652050621.py:81: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(pdfs, hdi_prob=hdi_prob, skipna=True)  # (T, 2)


/tmp/ipykernel_3881822/652050621.py:81: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(pdfs, hdi_prob=hdi_prob, skipna=True)  # (T, 2)
/tmp/ipykernel_3881822/652050621.py:81: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(pdfs, hdi_prob=hdi_prob, skipna=True)  # (T, 2)
/tmp/ipykernel_3881822/652050621.py:81: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(pdfs, hdi_prob=hdi_prob, skipna=True)  # (T, 2)


/tmp/ipykernel_3881822/652050621.py:81: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(pdfs, hdi_prob=hdi_prob, skipna=True)  # (T, 2)
/tmp/ipykernel_3881822/652050621.py:81: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(pdfs, hdi_prob=hdi_prob, skipna=True)  # (T, 2)
/tmp/ipykernel_3881822/652050621.py:81: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(pdfs, hdi_prob=hdi_prob, skipna=True)  # (T, 2)


/tmp/ipykernel_3881822/652050621.py:81: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(pdfs, hdi_prob=hdi_prob, skipna=True)  # (T, 2)
/tmp/ipykernel_3881822/652050621.py:81: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(pdfs, hdi_prob=hdi_prob, skipna=True)  # (T, 2)
/tmp/ipykernel_3881822/652050621.py:81: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(pdfs, hdi_prob=hdi_prob, skipna=True)  # (T, 2)


/tmp/ipykernel_3881822/652050621.py:81: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(pdfs, hdi_prob=hdi_prob, skipna=True)  # (T, 2)
/tmp/ipykernel_3881822/652050621.py:81: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(pdfs, hdi_prob=hdi_prob, skipna=True)  # (T, 2)
/tmp/ipykernel_3881822/652050621.py:81: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(pdfs, hdi_prob=hdi_prob, skipna=True)  # (T, 2)


...

/tmp/ipykernel_3881822/652050621.py:81: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(pdfs, hdi_prob=hdi_prob, skipna=True)  # (T, 2)
/tmp/ipykernel_3881822/652050621.py:81: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(pdfs, hdi_prob=hdi_prob, skipna=True)  # (T, 2)
/tmp/ipykernel_3881822/652050621.py:81: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(pdfs, hdi_prob=hdi_prob, skipna=True)  # (T, 2)


/tmp/ipykernel_3881822/652050621.py:81: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(pdfs, hdi_prob=hdi_prob, skipna=True)  # (T, 2)
/tmp/ipykernel_3881822/652050621.py:81: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(pdfs, hdi_prob=hdi_prob, skipna=True)  # (T, 2)
/tmp/ipykernel_3881822/652050621.py:81: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(pdfs, hdi_prob=hdi_prob, skipna=True)  # (T, 2)


/tmp/ipykernel_3881822/652050621.py:81: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(pdfs, hdi_prob=hdi_prob, skipna=True)  # (T, 2)
/tmp/ipykernel_3881822/652050621.py:81: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(pdfs, hdi_prob=hdi_prob, skipna=True)  # (T, 2)
/tmp/ipykernel_3881822/652050621.py:81: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(pdfs, hdi_prob=hdi_prob, skipna=True)  # (T, 2)


/tmp/ipykernel_3881822/652050621.py:81: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(pdfs, hdi_prob=hdi_prob, skipna=True)  # (T, 2)
/tmp/ipykernel_3881822/652050621.py:81: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(pdfs, hdi_prob=hdi_prob, skipna=True)  # (T, 2)
/tmp/ipykernel_3881822/652050621.py:81: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi = az.hdi(pdfs, hdi_prob=hdi_prob, skipna=True)  # (T, 2)


Saved figure to: figures/locust/order_density_polar_pdfs_grid.pdf


In [ ]:
# ============================================================
# Coefficient posteriors (fixed effects only), μ vs κ
# 95% HDI intervals + point estimate (mean)
# ============================================================

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import arviz as az

post = idata_hierarchical.posterior

TERMS = [
    ("Intercept",        "Intercept",               "kappa_Intercept"),
    ("Density",          "t_density",               "kappa_t_density"),
    ("Order",            "t_order",                 "kappa_t_order"),
    ("Density × Order",  "t_density:t_order",       "kappa_t_density:t_order"),
]

def _stack_1d(varname: str) -> np.ndarray:
    if varname not in post.data_vars:
        raise KeyError(
            f"Missing '{varname}' in idata.posterior. "
            f"Available: {list(post.data_vars)[:30]} ..."
        )
    return post[varname].stack(sample=("chain", "draw")).values.astype(float)

def circular_modulo(x):
    return (x + np.pi) % (2 * np.pi) - np.pi

def circular_mean(theta):
    z = np.exp(1j * np.asarray(theta, float))
    return float(circular_modulo(np.angle(z.mean())))

def circular_hdi(theta, hdi_prob=0.95):
    theta = np.asarray(theta, float)
    mu_c = circular_mean(theta)
    centered = circular_modulo(theta - mu_c)
    h = az.hdi(centered, hdi_prob=hdi_prob, skipna=True)
    low_c, high_c = float(h[0]), float(h[1])
    low = float(circular_modulo(low_c + mu_c))
    high = float(circular_modulo(high_c + mu_c))
    return low, high, mu_c

def linear_hdi_mean(x, hdi_prob=0.95):
    x = np.asarray(x, float)
    h = az.hdi(x, hdi_prob=hdi_prob, skipna=True)
    return float(x.mean()), float(h[0]), float(h[1])

rows = []
for pretty, mu_name, k_name in TERMS:
    mu_draws = _stack_1d(mu_name)
    k_draws  = _stack_1d(k_name)

    if mu_name == "Intercept":
        mu_lo, mu_hi, mu_m = circular_hdi(mu_draws, hdi_prob=0.95)
        rows.append(("mu", pretty, float(mu_m), float(mu_lo), float(mu_hi)))
    else:
        mu_m, mu_lo, mu_hi = linear_hdi_mean(mu_draws, hdi_prob=0.95)
        rows.append(("mu", pretty, float(mu_m), float(mu_lo), float(mu_hi)))

    k_m, k_lo, k_hi = linear_hdi_mean(k_draws, hdi_prob=0.95)
    rows.append(("kappa", pretty, float(k_m), float(k_lo), float(k_hi)))

terms_order = [t[0] for t in TERMS]
y_base = np.arange(len(terms_order), dtype=float)

mu_stats = {term: (mean, lo, hi) for (sub, term, mean, lo, hi) in rows if sub == "mu"}
k_stats  = {term: (mean, lo, hi) for (sub, term, mean, lo, hi) in rows if sub == "kappa"}

mu_mean = np.array([mu_stats[t][0] for t in terms_order], float)
mu_lo   = np.array([mu_stats[t][1] for t in terms_order], float)
mu_hi   = np.array([mu_stats[t][2] for t in terms_order], float)

k_mean = np.array([k_stats[t][0] for t in terms_order], float)
k_lo   = np.array([k_stats[t][1] for t in terms_order], float)
k_hi   = np.array([k_stats[t][2] for t in terms_order], float)

def circ_dist(a, b):
    return float(circular_modulo(b - a))

mu_xerr = np.zeros((2, len(terms_order)), float)
for i, term in enumerate(terms_order):
    if term == "Intercept":
        lo_w = abs(circ_dist(mu_lo[i], mu_mean[i]))
        hi_w = abs(circ_dist(mu_mean[i], mu_hi[i]))
        mu_xerr[:, i] = [lo_w, hi_w]
    else:
        mu_xerr[:, i] = [mu_mean[i] - mu_lo[i], mu_hi[i] - mu_mean[i]]

k_xerr = np.vstack([k_mean - k_lo, k_hi - k_mean])

dy = 0.16
y_mu = y_base - dy
y_k  = y_base + dy

tab10 = mpl.cm.get_cmap("tab10")
MU_COLOR = tab10(5)   # brown
K_COLOR  = tab10(2)   # green

# Figure: the matplotlib 'how' lives in locust_density_plots.py.
plots.plot_coefficient_posteriors(terms_order, mu_mean, mu_xerr, k_mean, k_xerr)


Saved figures to: figures/locust/order_density_coefficients.pdf; figures/locust/order_density_coefficients.png


/tmp/ipykernel_3881822/875676856.py:97: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  tab10 = mpl.cm.get_cmap("tab10")


In [ ]:
# ============================================================
# Single figure with 2 subplots (SWAPPED):
#   LEFT  = Model comparison (ΔELPD)
#   RIGHT = Model coefficients (μ vs κ)
# ============================================================

import ast
import textwrap
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import arviz as az

# ----------------------------
# Style
# ----------------------------
DPI = 300
AX_LW     = 0.8
TICK_LW   = 0.8
TICK_LEN  = 2.5
LINE_LW   = 1.0
ERR_LW    = 1.0
CAPSIZE   = 2.5
CAPTHICK  = 1.0
MS        = 4.5
MEW       = 0.8
FONT_SIZE = 7

mpl.rcParams.update({
    "font.size": FONT_SIZE,
    "axes.labelsize": FONT_SIZE,
    "axes.titlesize": FONT_SIZE,
    "legend.fontsize": FONT_SIZE,
    "xtick.labelsize": FONT_SIZE,
    "ytick.labelsize": FONT_SIZE,

    "axes.linewidth": AX_LW,
    "lines.linewidth": LINE_LW,

    "xtick.major.width": TICK_LW,
    "ytick.major.width": TICK_LW,
    "xtick.major.size": TICK_LEN,
    "ytick.major.size": TICK_LEN,

    "savefig.dpi": DPI,
    "figure.dpi": DPI,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

def _style_spines(ax):
    for s in ax.spines.values():
        s.set_linewidth(AX_LW)

def _finish_axis(ax, xlabel, yticks=None, yticklabels=None, legend_loc=None, vline0=True):
    if vline0:
        ax.axvline(0.0, linestyle="--", linewidth=ERR_LW, color="k", zorder=0)

    if yticks is not None:
        ax.set_yticks(yticks)
    if yticklabels is not None:
        ax.set_yticklabels(yticklabels)

    ax.invert_yaxis()
    ax.set_xlabel(xlabel)
    _style_spines(ax)

    if legend_loc is not None:
        leg = ax.legend(
            loc=legend_loc, frameon=True,
            handlelength=1.2, borderpad=0.3,
            labelspacing=0.3, handletextpad=0.4
        )
        leg.get_frame().set_linewidth(AX_LW)

tab10 = mpl.cm.get_cmap("tab10")
ELPD_INCL_COLOR = tab10(0)
ELPD_EXCL_COLOR = tab10(1)
MU_COLOR        = tab10(5)  # brown
K_COLOR         = tab10(2)  # green

# ----------------------------
# Model comparison data
# ----------------------------
RES_PATH = "mu_kappa_matched_model_sweep_pointwise_delta.parquet"
res_df = pd.read_parquet(RES_PATH)

def wrap_label(s, width=40):
    return textwrap.fill(str(s), width=width)

def _as_list(x):
    if x is None:
        return []
    if isinstance(x, (list, tuple)):
        return list(x)
    if isinstance(x, np.ndarray):
        return x.ravel().tolist()
    if isinstance(x, str):
        try:
            v = ast.literal_eval(x)
            if isinstance(v, np.ndarray):
                return v.ravel().tolist()
            if isinstance(v, (list, tuple)):
                return list(v)
        except Exception:
            return [x]
        return [x]
    try:
        if pd.isna(x):
            return []
    except Exception:
        pass
    return [str(x)]

res_df["has_order"] = res_df["terms"].apply(lambda t: any("t_order" in s for s in _as_list(t)))
res_df = res_df.sort_values("delta_elpd", ascending=False).reset_index(drop=True)

y_m = np.arange(len(res_df))
y_labels = [wrap_label(n, 40) for n in res_df["pretty_name"]]

g_incl = res_df[res_df["has_order"]].copy()
g_excl = res_df[~res_df["has_order"]].copy()
g_incl["y"] = g_incl.index.to_numpy()
g_excl["y"] = g_excl.index.to_numpy()

use_se = "se_delta_elpd" in res_df.columns and (not res_df["se_delta_elpd"].isna().all())

# ----------------------------
# Coefficient data
# ----------------------------
post = idata_hierarchical.posterior

TERMS = [
    ("Intercept",        "Intercept",               "kappa_Intercept"),
    ("Density",          "t_density",               "kappa_t_density"),
    ("Order",            "t_order",                 "kappa_t_order"),
    ("Density × Order",  "t_density:t_order",       "kappa_t_density:t_order"),
]

def _stack_1d(varname: str) -> np.ndarray:
    if varname not in post.data_vars:
        raise KeyError(
            f"Missing '{varname}' in idata.posterior. "
            f"Available: {list(post.data_vars)[:30]} ..."
        )
    return post[varname].stack(sample=("chain", "draw")).values.astype(float)

def circular_modulo(x):
    return (x + np.pi) % (2 * np.pi) - np.pi

def circular_mean(theta):
    z = np.exp(1j * np.asarray(theta, float))
    return float(circular_modulo(np.angle(z.mean())))

def circular_hdi(theta, hdi_prob=0.95):
    theta = np.asarray(theta, float)
    mu_c = circular_mean(theta)
    centered = circular_modulo(theta - mu_c)
    h = az.hdi(centered, hdi_prob=hdi_prob, skipna=True)
    low_c, high_c = float(h[0]), float(h[1])
    low = float(circular_modulo(low_c + mu_c))
    high = float(circular_modulo(high_c + mu_c))
    return low, high, mu_c

def linear_hdi_mean(x, hdi_prob=0.95):
    x = np.asarray(x, float)
    h = az.hdi(x, hdi_prob=hdi_prob, skipna=True)
    return float(x.mean()), float(h[0]), float(h[1])

def circ_dist(a, b):
    return float(circular_modulo(b - a))

rows = []
for pretty, mu_name, k_name in TERMS:
    mu_draws = _stack_1d(mu_name)
    k_draws  = _stack_1d(k_name)

    if mu_name == "Intercept":
        mu_lo, mu_hi, mu_m = circular_hdi(mu_draws, hdi_prob=0.95)
        rows.append(("mu", pretty, float(mu_m), float(mu_lo), float(mu_hi)))
    else:
        mu_m, mu_lo, mu_hi = linear_hdi_mean(mu_draws, hdi_prob=0.95)
        rows.append(("mu", pretty, float(mu_m), float(mu_lo), float(mu_hi)))

    k_m, k_lo, k_hi = linear_hdi_mean(k_draws, hdi_prob=0.95)
    rows.append(("kappa", pretty, float(k_m), float(k_lo), float(k_hi)))

terms_order = [t[0] for t in TERMS]
y_base = np.arange(len(terms_order), dtype=float)

mu_stats = {term: (mean, lo, hi) for (sub, term, mean, lo, hi) in rows if sub == "mu"}
k_stats  = {term: (mean, lo, hi) for (sub, term, mean, lo, hi) in rows if sub == "kappa"}

mu_mean = np.array([mu_stats[t][0] for t in terms_order], float)
mu_lo   = np.array([mu_stats[t][1] for t in terms_order], float)
mu_hi   = np.array([mu_stats[t][2] for t in terms_order], float)

k_mean = np.array([k_stats[t][0] for t in terms_order], float)
k_lo   = np.array([k_stats[t][1] for t in terms_order], float)
k_hi   = np.array([k_stats[t][2] for t in terms_order], float)

mu_xerr = np.zeros((2, len(terms_order)), float)
for i, term in enumerate(terms_order):
    if term == "Intercept":
        lo_w = abs(circ_dist(mu_lo[i], mu_mean[i]))
        hi_w = abs(circ_dist(mu_mean[i], mu_hi[i]))
        mu_xerr[:, i] = [lo_w, hi_w]
    else:
        mu_xerr[:, i] = [mu_mean[i] - mu_lo[i], mu_hi[i] - mu_mean[i]]

k_xerr = np.vstack([k_mean - k_lo, k_hi - k_mean])

dy = 0.16
y_mu = y_base - dy
y_k  = y_base + dy

# ----------------------------
# Draw
# ----------------------------
ROW_H = 0.36
n_rows = max(len(res_df), len(terms_order))
fig_h = ROW_H * n_rows

# Figure: the matplotlib 'how' lives in locust_density_plots.py.
plots.plot_model_selection_and_coefficients(
    g_incl, g_excl, y_m, y_labels, use_se,
    terms_order, y_base, y_mu, y_k,
    mu_mean, mu_xerr, k_mean, k_xerr
)

print(res_df[["pretty_name", "elpd", "delta_elpd", "se_delta_elpd", "pareto_k_max", "best_model"]].to_string(index=False))


/tmp/ipykernel_3881822/1474518968.py:77: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  tab10 = mpl.cm.get_cmap("tab10")


Saved figures to: figures/locust/order_density_model_selection_and_coefficients.pdf; figures/locust/order_density_model_selection_and_coefficients.png


    pretty_name        elpd  delta_elpd  se_delta_elpd  pareto_k_max      best_model
Density × Order -590.520947    0.000000       0.000000      0.317989 Density × Order
Density + Order -592.963440   -2.442493       2.612576      0.226814 Density × Order
          Order -599.800766   -9.279819       5.739002      0.102140 Density × Order
        Density -666.417847  -75.896900      11.557480      0.131385 Density × Order
 Intercept-only -669.203208  -78.682261      11.612378      0.005632 Density × Order


In [ ]:
# ============================================================
# FULL UPDATED CODE (SMALL MULTIPLES) — with PDF export
# Each line gets its own subplot + its matching bootstrap points/intervals.
# Layout: 2 columns
#   Left column:  alignment vs Order, one row per Density level
#   Right column: alignment vs Density, one row per Order level
# ============================================================

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import arviz as az
from scipy.special import i0, i1

from joblib import Parallel, delayed
from tqdm.notebook import tqdm
import contextlib
import joblib

from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from matplotlib.legend_handler import HandlerBase


# ----------------------------
# Global style (kept consistent with your other figures)
# ----------------------------
DPI = 300
AX_LW     = 0.8
TICK_LW   = 0.8
TICK_LEN  = 2.5

mpl.rcParams.update({
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 7,
    "legend.fontsize": 7,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,

    "axes.linewidth": AX_LW,
    "xtick.major.width": TICK_LW,
    "ytick.major.width": TICK_LW,
    "xtick.major.size": TICK_LEN,
    "ytick.major.size": TICK_LEN,

    "savefig.dpi": DPI,
    "figure.dpi": DPI,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


class HandlerPatchOverLine(HandlerBase):
    def create_artists(self, legend, orig_handle, xdescent, ydescent, width, height,
                       fontsize, trans):
        band, line = orig_handle  # (Patch, Line2D)

        r = Rectangle(
            (xdescent, ydescent), width, height,
            facecolor=band.get_facecolor(),
            edgecolor="none",
            alpha=band.get_alpha(),
            transform=trans
        )

        l = Line2D(
            [xdescent, xdescent + width],
            [ydescent + 0.5 * height, ydescent + 0.5 * height],
            color=line.get_color(),
            lw=line.get_linewidth(),
            transform=trans
        )

        return [r, l]


# ----------------------------
# Transforms used in the model
# ----------------------------
def normalize(x):
    return x * 2.0 - 1.0

def log2_scale(x, min_size=1., max_size=64.):
    return np.log2(x / min_size) / np.log2(max_size / min_size)

DENS_MIN, DENS_MAX = 1.0, 64.0

def softplus(x):
    x = np.asarray(x)
    return np.where(x > 20, x, np.log1p(np.exp(x)))

def circular_modulo(x):
    return (x + np.pi) % (2 * np.pi) - np.pi

def A_of_kappa(kappa):
    return i1(kappa) / i0(kappa)


# ----------------------------
# Posterior linear predictors
# ----------------------------
def posterior_linpred(idata, prefix, X_df):
    post = idata.posterior

    if prefix:
        coefs = [v for v in post.data_vars if v.startswith(prefix)]
        key_intercept = f"{prefix}Intercept"
    else:
        coefs = [v for v in post.data_vars if (not v.startswith("kappa_"))]
        key_intercept = "Intercept"

    if key_intercept not in post.data_vars:
        raise ValueError(f"Intercept '{key_intercept}' not found in posterior.")

    def flat(name):
        return post[name].stack(sample=("chain", "draw")).values  # (S,)

    n = len(X_df)
    eta = flat(key_intercept)[:, None] * np.ones((1, n), dtype=float)

    for coef in coefs:
        if coef == key_intercept:
            continue
        col = coef.replace(prefix, "")
        if col in X_df.columns:
            eta += flat(coef)[:, None] * X_df[col].to_numpy()[None, :]

    return eta  # (S, n)


# -----------------------------
# Bambi-style centering
# -----------------------------
def _compute_training_means(merged_df):
    t_order = normalize(merged_df["order"].astype(float).to_numpy())
    t_density = normalize(
        log2_scale(merged_df["density"].astype(float).to_numpy(), DENS_MIN, DENS_MAX)
    )
    return {"t_order": np.mean(t_order), "t_density": np.mean(t_density)}

TRAIN_MEANS = _compute_training_means(merged_df)

def make_X(order_vals, density_vals, idata, center_predictors=True, interaction_from_centered=True):
    X = pd.DataFrame({"order": order_vals, "density": density_vals})
    X["t_order"]   = normalize(X["order"].astype(float))
    X["t_density"] = normalize(log2_scale(X["density"].astype(float), DENS_MIN, DENS_MAX))

    if center_predictors:
        X["t_order"]   = X["t_order"]   - TRAIN_MEANS["t_order"]
        X["t_density"] = X["t_density"] - TRAIN_MEANS["t_density"]

    has_interaction = (
        ("t_density:t_order" in idata.posterior.data_vars) or
        ("kappa_t_density:t_order" in idata.posterior.data_vars)
    )
    if has_interaction:
        if interaction_from_centered:
            X["t_density:t_order"] = X["t_density"] * X["t_order"]
        else:
            X_raw_order   = normalize(pd.Series(order_vals).astype(float))
            X_raw_density = normalize(
                log2_scale(pd.Series(density_vals).astype(float), DENS_MIN, DENS_MAX)
            )
            X["t_density:t_order"] = np.asarray(X_raw_density) * np.asarray(X_raw_order)

    return X

def posterior_alignment(idata, X, hdi_prob=0.95):
    eta_mu    = posterior_linpred(idata, prefix="",       X_df=X)
    eta_kappa = posterior_linpred(idata, prefix="kappa_", X_df=X)

    mu    = circular_modulo(eta_mu)
    kappa = softplus(eta_kappa)

    align = A_of_kappa(kappa) * np.cos(mu)  # (S, n)

    mean = align.mean(axis=0)
    hdi  = az.hdi(align, hdi_prob=hdi_prob, skipna=True)  # (n,2)
    return {"mean": mean, "low": hdi[:, 0], "high": hdi[:, 1]}


# ============================================================
# joblib <-> tqdm glue
# ============================================================
@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_cb = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_cb
        tqdm_object.close()


# ============================================================
# Empirical hierarchical bootstrap per treatment cell (parallel)
# ============================================================
def hierarchical_bootstrap_cells_parallel(
    df_steps,
    B=1000,
    ci=0.95,
    seed=123,
    n_jobs=-1,
    prefer="threads",
    verbose=0,
):
    rng_master = np.random.default_rng(seed)

    records = []
    for aid, g in df_steps.groupby("animal_id", observed=True):
        dens = float(g["density"].iloc[0])
        ordv = float(g["order"].iloc[0])
        u = np.exp(1j * g["y"].to_numpy())
        records.append({"animal_id": aid, "density": dens, "order": ordv, "u": u})
    animal_tbl = pd.DataFrame(records)

    cell_groups = {}
    for (dens, ordv), sub in animal_tbl.groupby(["density", "order"], observed=True):
        cell_groups[(float(dens), float(ordv))] = sub["animal_id"].to_numpy()

    u_by_animal = dict(zip(animal_tbl["animal_id"], animal_tbl["u"]))

    def _hdi_1d(x, prob):
        h = az.hdi(x, hdi_prob=prob, skipna=True)
        h = np.asarray(h)
        if h.ndim == 2:
            return float(h[0, 0]), float(h[0, 1])
        return float(h[0]), float(h[1])

    def _bootstrap_one_cell(cell_key, animals, cell_seed):
        rng = np.random.default_rng(cell_seed)
        animals = np.asarray(animals)
        nA = len(animals)
        boots = np.empty(B, dtype=float)

        for b in range(B):
            samp_animals = animals[rng.integers(0, nA, size=nA)]
            z_sum = 0.0 + 0.0j

            for aid in samp_animals:
                u = u_by_animal[aid]
                idx = rng.integers(0, len(u), size=len(u))
                z_sum += u[idx].mean()

            boots[b] = np.real(z_sum / nA)

        low, high = _hdi_1d(boots, ci)

        return {
            "density": cell_key[0],
            "order": cell_key[1],
            "n_animals": int(nA),
            "emp_mean": float(boots.mean()),
            "emp_low":  float(low),
            "emp_high": float(high),
        }

    cells = list(cell_groups.items())
    cell_seeds = rng_master.integers(0, 2**32 - 1, size=len(cells), dtype=np.uint32)

    with tqdm_joblib(tqdm(total=len(cells), desc="Hierarchical bootstrap (cells)")):
        out = Parallel(n_jobs=n_jobs, prefer=prefer, verbose=verbose)(
            delayed(_bootstrap_one_cell)(cell_key, animals, int(cell_seeds[i]))
            for i, (cell_key, animals) in enumerate(cells)
        )

    return pd.DataFrame(out)


# ============================================================
# Build grids, compute model predictions, compute bootstrap
# ============================================================
# DESCENDING so subplot rows run largest -> smallest (matching your other figures)
dens_levels  = np.sort(merged_df["density"].unique())[::-1]
order_levels = np.sort(merged_df["order"].unique())[::-1]

order_grid   = np.linspace(order_levels.min(), order_levels.max(), 80)
density_grid = np.geomspace(max(dens_levels.min(), 1e-6), dens_levels.max(), 80)

pred_i = {}
for d in dens_levels:
    X = make_X(
        order_grid,
        np.full_like(order_grid, d, dtype=float),
        idata_hierarchical,
        center_predictors=True,
        interaction_from_centered=True,
    )
    keep_cols = ["t_order", "t_density"] + (["t_density:t_order"] if "t_density:t_order" in X.columns else [])
    pred_i[d] = posterior_alignment(idata_hierarchical, X[keep_cols], hdi_prob=0.95)

pred_ii = {}
for o in order_levels:
    X = make_X(
        np.full_like(density_grid, o, dtype=float),
        density_grid,
        idata_hierarchical,
        center_predictors=True,
        interaction_from_centered=True,
    )
    keep_cols = ["t_order", "t_density"] + (["t_density:t_order"] if "t_density:t_order" in X.columns else [])
    pred_ii[o] = posterior_alignment(idata_hierarchical, X[keep_cols], hdi_prob=0.95)

emp = hierarchical_bootstrap_cells_parallel(
    df,
    B=1000,
    ci=0.95,
    seed=123,
    n_jobs=-1,
    prefer="threads",
)

def emp_for_density(d):
    # keep within-panel x ordering increasing (order 0 -> 1)
    sub = emp.loc[np.isclose(emp["density"], float(d))].sort_values("order")
    return None if sub.empty else sub

def emp_for_order(o):
    # keep within-panel x ordering increasing (density 1 -> 64)
    sub = emp.loc[np.isclose(emp["order"], float(o))].sort_values("density")
    return None if sub.empty else sub


# ============================================================
# SMALL MULTIPLES PLOT
# ============================================================
n_rows = max(len(dens_levels), len(order_levels))

fig_h = max(2.2, 1.25 * n_rows)

# Figure: the matplotlib 'how' lives in locust_density_plots.py.
plots.plot_alignment_ppc_bootstrap(
    dens_levels, order_levels, order_grid, density_grid, pred_i, pred_ii, emp
)


/tmp/ipykernel_3881822/3758735196.py:180: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi  = az.hdi(align, hdi_prob=hdi_prob, skipna=True)  # (n,2)
/tmp/ipykernel_3881822/3758735196.py:180: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi  = az.hdi(align, hdi_prob=hdi_prob, skipna=True)  # (n,2)
/tmp/ipykernel_3881822/3758735196.py:180: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi  = az.hdi(align, hdi_prob=hdi_prob, skipna=True)  # (n,2)


/tmp/ipykernel_3881822/3758735196.py:180: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi  = az.hdi(align, hdi_prob=hdi_prob, skipna=True)  # (n,2)
/tmp/ipykernel_3881822/3758735196.py:180: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi  = az.hdi(align, hdi_prob=hdi_prob, skipna=True)  # (n,2)
/tmp/ipykernel_3881822/3758735196.py:180: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi  = az.hdi(align, hdi_prob=hdi_prob, skipna=True)  # (n,2)


/tmp/ipykernel_3881822/3758735196.py:180: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi  = az.hdi(align, hdi_prob=hdi_prob, skipna=True)  # (n,2)
/tmp/ipykernel_3881822/3758735196.py:180: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi  = az.hdi(align, hdi_prob=hdi_prob, skipna=True)  # (n,2)
/tmp/ipykernel_3881822/3758735196.py:180: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi  = az.hdi(align, hdi_prob=hdi_prob, skipna=True)  # (n,2)


/tmp/ipykernel_3881822/3758735196.py:180: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi  = az.hdi(align, hdi_prob=hdi_prob, skipna=True)  # (n,2)
/tmp/ipykernel_3881822/3758735196.py:180: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi  = az.hdi(align, hdi_prob=hdi_prob, skipna=True)  # (n,2)
/tmp/ipykernel_3881822/3758735196.py:180: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi  = az.hdi(align, hdi_prob=hdi_prob, skipna=True)  # (n,2)


notebooks/locust_density_plots.py:1128: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


Saved figure to: figures/locust/order_density_alignment_ppc_bootstrap.pdf


In [ ]:
# ============================================================
# SMALL-MULTIPLES (DOTS + ERRORBARS, MODEL vs DATA at observed cells)
#   - Data: hierarchical bootstrap per observed (density, order) cell
#   - Model: posterior predictive sampled from individual-level likelihoods
#            + same hierarchical bootstrap procedure (animals + steps)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arviz as az
from math import pi

from joblib import Parallel, delayed
from tqdm.notebook import tqdm
import contextlib
import joblib

from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from matplotlib.legend_handler import HandlerBase


# ----------------------------
# transforms used in the model
# ----------------------------
def normalize(x):
    return np.asarray(x, dtype=float) * 2.0 - 1.0

def log2_scale(x, min_size=1.0, max_size=64.0):
    x = np.asarray(x, dtype=float)
    return np.log2(x / min_size) / np.log2(max_size / min_size)

DENS_MIN, DENS_MAX = 1.0, 64.0

def softplus(x):
    x = np.asarray(x)
    return np.where(x > 20, x, np.log1p(np.exp(x)))

def circular_modulo(x):
    return (x + np.pi) % (2 * np.pi) - np.pi

def alignment_stat(y):
    y = np.asarray(y, dtype=float)
    return float(np.real(np.exp(1j * y).mean()))


# -----------------------------
# Bambi-style centering
# -----------------------------
def _compute_training_means(merged_df):
    t_order = normalize(merged_df["order"].astype(float).to_numpy())
    t_density = normalize(log2_scale(merged_df["density"].astype(float).to_numpy(), DENS_MIN, DENS_MAX))
    return {"t_order": float(np.mean(t_order)), "t_density": float(np.mean(t_density))}

TRAIN_MEANS = _compute_training_means(merged_df)


# ============================================================
# joblib <-> tqdm glue
# ============================================================
@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_cb = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_cb
        tqdm_object.close()


# ============================================================
# Empirical hierarchical bootstrap per treatment cell (parallel)
# ============================================================
def hierarchical_bootstrap_cells_parallel(
    df_steps,
    B=1000,
    ci=0.95,              # interpreted as HDI probability mass
    seed=123,
    n_jobs=-1,
    prefer="threads",
    verbose=0,
):
    rng_master = np.random.default_rng(seed)

    records = []
    for aid, g in df_steps.groupby("animal_id", observed=True):
        dens = float(g["density"].iloc[0])
        ordv = float(g["order"].iloc[0])
        u = np.exp(1j * g["y"].to_numpy())
        records.append({"animal_id": str(aid), "density": dens, "order": ordv, "u": u})
    animal_tbl = pd.DataFrame(records)

    cell_groups = {}
    for (dens, ordv), sub in animal_tbl.groupby(["density", "order"], observed=True):
        cell_groups[(float(dens), float(ordv))] = sub["animal_id"].to_numpy()

    u_by_animal = dict(zip(animal_tbl["animal_id"], animal_tbl["u"]))

    def _hdi_1d(x, prob):
        h = az.hdi(x, hdi_prob=prob, skipna=True)
        h = np.asarray(h)
        if h.ndim == 2:
            return float(h[0, 0]), float(h[0, 1])
        return float(h[0]), float(h[1])

    def _bootstrap_one_cell(cell_key, animals, cell_seed):
        rng = np.random.default_rng(cell_seed)
        animals = np.asarray(animals)
        nA = len(animals)
        boots = np.empty(B, dtype=float)

        for b in range(B):
            samp_animals = animals[rng.integers(0, nA, size=nA)]
            z_sum = 0.0 + 0.0j

            for aid in samp_animals:
                u = u_by_animal[aid]
                idx = rng.integers(0, len(u), size=len(u))
                z_sum += u[idx].mean()

            boots[b] = np.real(z_sum / nA)

        low, high = _hdi_1d(boots, ci)

        return {
            "density": cell_key[0],
            "order": cell_key[1],
            "n_animals": int(nA),
            "emp_mean": float(boots.mean()),
            "emp_low":  float(low),
            "emp_high": float(high),
        }

    cells = list(cell_groups.items())
    cell_seeds = rng_master.integers(0, 2**32 - 1, size=len(cells), dtype=np.uint32)

    with tqdm_joblib(tqdm(total=len(cells), desc="Hierarchical bootstrap (data cells)")):
        out = Parallel(n_jobs=n_jobs, prefer=prefer, verbose=verbose)(
            delayed(_bootstrap_one_cell)(cell_key, animals, int(cell_seeds[i]))
            for i, (cell_key, animals) in enumerate(cells)
        )

    return pd.DataFrame(out)


# ============================================================
# Model: individual-level likelihood sampling + same hierarchical bootstrap
# ============================================================
def ppc_hierarchical_bootstrap_cells_parallel(
    df_steps,
    merged_df,
    idata,
    B=600,
    ci=0.95,
    seed=2025,
    n_jobs=-1,
    prefer="threads",
    verbose=0,
    bootstrap_steps=True,
):
    rng_master = np.random.default_rng(seed)
    post = idata.posterior

    animal_dim = None
    for d in post.dims:
        if "animal_id" in d:
            animal_dim = d
            break
    if animal_dim is None:
        raise KeyError(f"Could not find an animal dimension containing 'animal_id'. Found dims: {list(post.dims)}")

    def stack_samples_1d(name, default_zero=False):
        if name not in post.data_vars:
            if default_zero:
                S = int(post.sizes["chain"] * post.sizes["draw"])
                return np.zeros(S, dtype=float)
            raise KeyError(f"Missing posterior variable: {name}")
        return post[name].stack(sample=("chain", "draw")).values.astype(float)

    def stack_samples_re(name):
        if name not in post.data_vars:
            raise KeyError(f"Missing posterior random effect: {name}")
        da = post[name].stack(sample=("chain", "draw")).transpose("sample", animal_dim)
        return da.values.astype(float)

    b0    = stack_samples_1d("Intercept")
    b_td  = stack_samples_1d("t_density", default_zero=True)
    b_to  = stack_samples_1d("t_order", default_zero=True)
    b_int = stack_samples_1d("t_density:t_order", default_zero=True)

    k0    = stack_samples_1d("kappa_Intercept")
    k_td  = stack_samples_1d("kappa_t_density", default_zero=True)
    k_to  = stack_samples_1d("kappa_t_order", default_zero=True)
    k_int = stack_samples_1d("kappa_t_density:t_order", default_zero=True)

    if "1|animal_id" in post.data_vars:
        mu_re = stack_samples_re("1|animal_id")
    else:
        # pooled-mu spec: direction has no per-animal effect
        S_tot = int(post.sizes["chain"] * post.sizes["draw"])
        mu_re = np.zeros((S_tot, int(post.sizes[animal_dim])), dtype=float)
    kappa_re = stack_samples_re("kappa_1|animal_id")

    S_total = b0.shape[0]
    post_animal_ids = post.coords[animal_dim].values.astype(str)
    post_index = {aid: i for i, aid in enumerate(post_animal_ids)}

    df_local = df_steps.copy()
    df_local["animal_id"] = df_local["animal_id"].astype(str)

    animal_tbl = (
        df_local.groupby("animal_id", observed=True)
        .agg(density=("density", "first"),
             order=("order", "first"),
             n_steps=("y", "size"))
        .reset_index()
    )

    animal_tbl["t_order"]   = normalize(animal_tbl["order"].astype(float)) - TRAIN_MEANS["t_order"]
    animal_tbl["t_density"] = normalize(log2_scale(animal_tbl["density"].astype(float), DENS_MIN, DENS_MAX)) - TRAIN_MEANS["t_density"]
    animal_tbl["t_int"]     = animal_tbl["t_order"] * animal_tbl["t_density"]

    animal_tbl["post_idx"] = animal_tbl["animal_id"].map(post_index)
    if animal_tbl["post_idx"].isna().any():
        bad = animal_tbl.loc[animal_tbl["post_idx"].isna(), "animal_id"].tolist()[:10]
        raise ValueError(
            "Some df animal_id levels not found in posterior coords. "
            f"Examples: {bad}\nPosterior coord example: {post_animal_ids[:10].tolist()}"
        )
    animal_tbl["post_idx"] = animal_tbl["post_idx"].astype(int)

    dens_arr     = animal_tbl["density"].to_numpy(float)
    order_arr    = animal_tbl["order"].to_numpy(float)
    nsteps_arr   = animal_tbl["n_steps"].to_numpy(int)
    torder_arr   = animal_tbl["t_order"].to_numpy(float)
    tdens_arr    = animal_tbl["t_density"].to_numpy(float)
    tint_arr     = animal_tbl["t_int"].to_numpy(float)
    postidx_arr  = animal_tbl["post_idx"].to_numpy(int)

    cell_to_rows = {}
    for i in range(len(animal_tbl)):
        key = (float(dens_arr[i]), float(order_arr[i]))
        cell_to_rows.setdefault(key, []).append(i)

    def _hdi_1d(x, prob):
        h = az.hdi(x, hdi_prob=prob, skipna=True)
        h = np.asarray(h)
        if h.ndim == 2:
            return float(h[0, 0]), float(h[0, 1])
        return float(h[0]), float(h[1])

    def _one_cell(cell_key, row_idx_list, cell_seed):
        rng = np.random.default_rng(cell_seed)
        rows = np.asarray(row_idx_list, dtype=int)
        nA = len(rows)
        boots = np.empty(B, dtype=float)

        for b in range(B):
            samp_rows = rows[rng.integers(0, nA, size=nA)]
            s = int(rng.integers(0, S_total))

            z_sum = 0.0 + 0.0j
            for rr in samp_rows:
                j = int(postidx_arr[rr])
                n_steps = int(nsteps_arr[rr])

                eta_mu = (
                    b0[s]
                    + b_td[s]  * tdens_arr[rr]
                    + b_to[s]  * torder_arr[rr]
                    + b_int[s] * tint_arr[rr]
                    + mu_re[s, j]
                )
                eta_kappa = (
                    k0[s]
                    + k_td[s]  * tdens_arr[rr]
                    + k_to[s]  * torder_arr[rr]
                    + k_int[s] * tint_arr[rr]
                    + kappa_re[s, j]
                )

                mu = circular_modulo(eta_mu)
                kappa = softplus(eta_kappa)

                y_rep = rng.vonmises(mu, kappa, size=n_steps)
                u = np.exp(1j * y_rep)

                if bootstrap_steps:
                    idx = rng.integers(0, n_steps, size=n_steps)
                    z_sum += u[idx].mean()
                else:
                    z_sum += u.mean()

            boots[b] = float(np.real(z_sum / nA))

        low, high = _hdi_1d(boots, ci)
        return {
            "density": cell_key[0],
            "order": cell_key[1],
            "n_animals": int(nA),
            "model_mean": float(boots.mean()),
            "model_low":  float(low),
            "model_high": float(high),
        }

    cells = list(cell_to_rows.items())
    cell_seeds = rng_master.integers(0, 2**32 - 1, size=len(cells), dtype=np.uint32)

    with tqdm_joblib(tqdm(total=len(cells), desc="Hierarchical bootstrap (model cells)")):
        out = Parallel(n_jobs=n_jobs, prefer=prefer, verbose=verbose)(
            delayed(_one_cell)(cell_key, row_idx_list, int(cell_seeds[i]))
            for i, (cell_key, row_idx_list) in enumerate(cells)
        )

    return pd.DataFrame(out)


# ============================================================
# Compute bootstrap summaries (data + model)
# ============================================================
emp = hierarchical_bootstrap_cells_parallel(
    df,
    B=1000,
    ci=0.95,
    seed=123,
    n_jobs=-1,
    prefer="threads",
)

mod = ppc_hierarchical_bootstrap_cells_parallel(
    df,
    merged_df,
    idata_hierarchical,
    B=600,
    ci=0.95,
    seed=2025,
    n_jobs=-1,
    prefer="threads",
    bootstrap_steps=True,
)

# DESCENDING so subplot rows run largest -> smallest
dens_levels  = np.sort(merged_df["density"].unique())[::-1]
order_levels = np.sort(merged_df["order"].unique())[::-1]


# ============================================================
# SMALL MULTIPLES PLOT (DOTS + ERRORBARS)
# ============================================================
plt.rcParams.update({
    "font.size": 8,
    "ytick.major.width": 1,
    "xtick.major.width": 1,
    "axes.linewidth": 1,
    "font.family": "Arial"
})

n_rows = max(len(dens_levels), len(order_levels))
fig_h = max(2.2, 1.25 * n_rows)

# Figure: the matplotlib 'how' lives in locust_density_plots.py.
plots.plot_cell_ppc_dots(dens_levels, order_levels, emp, mod)


notebooks/locust_density_plots.py:1275: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


Saved figures to: figures/locust/order_density_cell_ppc_dots.pdf; figures/locust/order_density_cell_ppc_dots.png


In [ ]:
# ============================================================
# INDIVIDUAL-LEVEL PPC BY TREATMENT CELL (density × order)
# FULL UPDATED CODE (with correct RE stacking order)
#
# Each subplot = one (density, order) combination.
# x-axis = individual (animal) within that cell
# y-axis = alignment statistic: Re(mean(exp(i*y)))
#
# For each individual:
#   - Data: within-individual bootstrap CI (resample steps within that animal)
#   - Model: posterior predictive CI (simulate same N steps using that animal's RE + fixed)
#
# Requires in memory:
#   merged_df : original df with columns density, order
#   df        : step-level df used for fitting, columns: animal_id, y, density, order
#   idata_hierarchical : fitted InferenceData from bambi
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arviz as az

# ============================================================
# SETTINGS
# ============================================================
DENS_MIN, DENS_MAX = 1.0, 64.0
HDP = 0.95

B_BOOT = 800          # within-animal bootstrap reps (data)
BOOT_SEED = 123

S_PPC = 400           # posterior draws per animal for PPC reps (model)
PPC_SEED = 2025       # base seed for PPC

ANIMAL_DIM = "animal_id__factor_dim"  # from your idata.posterior dims

# ============================================================
# TRANSFORMS (match your model)
# ============================================================
def normalize(x):
    return np.asarray(x, dtype=float) * 2.0 - 1.0

def log2_scale(x, min_size=1., max_size=64.):
    x = np.asarray(x, dtype=float)
    return np.log2(x / min_size) / np.log2(max_size / min_size)

def softplus(x):
    x = np.asarray(x)
    return np.where(x > 20, x, np.log1p(np.exp(x)))

def circular_modulo(x):
    return (x + np.pi) % (2 * np.pi) - np.pi

def alignment_stat(y):
    y = np.asarray(y, dtype=float)
    return float(np.real(np.exp(1j * y).mean()))

# ============================================================
# TRAINING MEANS (center_predictors=True)
# ============================================================
def compute_training_means(merged_df):
    t_order = normalize(merged_df["order"].astype(float).to_numpy())
    t_density = normalize(log2_scale(merged_df["density"].astype(float).to_numpy(), DENS_MIN, DENS_MAX))
    return {"t_order": float(np.mean(t_order)), "t_density": float(np.mean(t_density))}

TRAIN_MEANS = compute_training_means(merged_df)

# ============================================================
# WITHIN-ANIMAL BOOTSTRAP (DATA)
# ============================================================
def bootstrap_alignment(y, B=800, seed=123):
    rng = np.random.default_rng(seed)
    u = np.exp(1j * np.asarray(y, dtype=float))
    n = len(u)
    boots = np.empty(B, dtype=float)
    for b in range(B):
        idx = rng.integers(0, n, size=n)
        boots[b] = float(np.real(u[idx].mean()))
    return boots

# ============================================================
# POSTERIOR ACCESS (your parameter names)
# ============================================================
post = idata_hierarchical.posterior
if ANIMAL_DIM not in post.dims:
    raise KeyError(f"Expected dim '{ANIMAL_DIM}' not in posterior. Found dims: {list(post.dims)}")

def stack_samples_1d(da):
    """For fixed effects: dims (chain, draw) -> (S,)"""
    return da.stack(sample=("chain", "draw")).values

def stack_samples_re(da, animal_dim):
    """
    For random effects: ensure output is (S, n_animals) no matter how dims are ordered.
    """
    da2 = da.stack(sample=("chain", "draw")).transpose("sample", animal_dim)
    return da2.values

# fixed effects (mu)
b0    = stack_samples_1d(post["Intercept"])                 # (S,)
b_td  = stack_samples_1d(post["t_density"])                 # (S,)
b_to  = stack_samples_1d(post["t_order"])                   # (S,)
b_int = stack_samples_1d(post["t_density:t_order"])         # (S,)

# fixed effects (kappa pre-softplus)
k0    = stack_samples_1d(post["kappa_Intercept"])           # (S,)
k_td  = stack_samples_1d(post["kappa_t_density"])           # (S,)
k_to  = stack_samples_1d(post["kappa_t_order"])             # (S,)
k_int = stack_samples_1d(post["kappa_t_density:t_order"])   # (S,)

# random intercepts (already present; stack + transpose correctly!)
if "1|animal_id" in post.data_vars:
    mu_re    = stack_samples_re(post["1|animal_id"], ANIMAL_DIM)     # (S, n_animals)
else:
    # pooled-mu spec: direction has no per-animal effect
    mu_re    = np.zeros((b0.shape[0], int(post.sizes[ANIMAL_DIM])), dtype=float)
kappa_re = stack_samples_re(post["kappa_1|animal_id"], ANIMAL_DIM)   # (S, n_animals)

S_total = b0.shape[0]
n_animals_post = mu_re.shape[1]
print(f"[OK] posterior draws S={S_total}, animals={n_animals_post}")
print(f"[OK] mu_re shape={mu_re.shape} kappa_re shape={kappa_re.shape}")

# ============================================================
# BUILD 1-ROW-PER-ANIMAL TABLE (each animal has 1 treatment)
# ============================================================
df_local = df.copy()
df_local["animal_id"] = df_local["animal_id"].astype(str)

y_by_animal = {aid: g["y"].to_numpy() for aid, g in df_local.groupby("animal_id", observed=True)}

animal_tbl = (
    df_local.groupby("animal_id", observed=True)
    .agg(density=("density", "first"), order=("order", "first"), n_steps=("y", "size"))
    .reset_index()
)

# centered predictors used in model
animal_tbl["t_order"]   = normalize(animal_tbl["order"].astype(float)) - TRAIN_MEANS["t_order"]
animal_tbl["t_density"] = normalize(log2_scale(animal_tbl["density"].astype(float), DENS_MIN, DENS_MAX)) - TRAIN_MEANS["t_density"]
animal_tbl["t_int"]     = animal_tbl["t_order"] * animal_tbl["t_density"]

# map animal_id -> posterior index using coords on ANIMAL_DIM
post_animal_ids = post.coords[ANIMAL_DIM].values.astype(str)
post_index = {aid: i for i, aid in enumerate(post_animal_ids)}

animal_tbl["post_idx"] = animal_tbl["animal_id"].map(post_index)
if animal_tbl["post_idx"].isna().any():
    bad = animal_tbl.loc[animal_tbl["post_idx"].isna(), "animal_id"].tolist()[:10]
    raise ValueError(
        "Some df animal_id levels not found in posterior coords. "
        f"Examples: {bad}\nPosterior coord example: {post_animal_ids[:10].tolist()}"
    )
animal_tbl["post_idx"] = animal_tbl["post_idx"].astype(int)

# ============================================================
# COMPUTE DATA BOOTSTRAP + MODEL PPC (per animal)
# ============================================================
rng_master = np.random.default_rng(PPC_SEED)
out_rows = []

for row in animal_tbl.itertuples(index=False):
    aid = row.animal_id
    j = int(row.post_idx)
    n_steps = int(row.n_steps)
    y_obs = y_by_animal[aid]

    # ---- DATA bootstrap ----
    boots = bootstrap_alignment(y_obs, B=B_BOOT, seed=BOOT_SEED + j)
    data_mean = float(boots.mean())
    data_low, data_high = az.hdi(boots, hdi_prob=HDP, skipna=True)
    data_low, data_high = float(data_low), float(data_high)

    # ---- MODEL PPC ----
    draw_idx = rng_master.choice(S_total, size=min(S_PPC, S_total), replace=False)

    eta_mu = (
        b0[draw_idx]
        + b_td[draw_idx]  * float(row.t_density)
        + b_to[draw_idx]  * float(row.t_order)
        + b_int[draw_idx] * float(row.t_int)
        + mu_re[draw_idx, j]
    )

    eta_kappa = (
        k0[draw_idx]
        + k_td[draw_idx]  * float(row.t_density)
        + k_to[draw_idx]  * float(row.t_order)
        + k_int[draw_idx] * float(row.t_int)
        + kappa_re[draw_idx, j]
    )

    mu = circular_modulo(eta_mu)
    kappa = softplus(eta_kappa)

    rng = np.random.default_rng(PPC_SEED + 10_000 + j)
    reps = np.empty(len(draw_idx), dtype=float)
    for i in range(len(draw_idx)):
        y_rep = rng.vonmises(mu[i], kappa[i], size=n_steps)
        reps[i] = alignment_stat(y_rep)

    model_mean = float(reps.mean())
    model_low, model_high = az.hdi(reps, hdi_prob=HDP, skipna=True)

    out_rows.append({
        "animal_id": aid,
        "density": float(row.density),
        "order": float(row.order),
        "data_mean": data_mean,
        "data_low": float(data_low),
        "data_high": float(data_high),
        "model_mean": model_mean,
        "model_low": float(model_low),
        "model_high": float(model_high),
    })

ppc_df = pd.DataFrame(out_rows)

# ============================================================
# PLOT: grid of subplots by (density, order)
# ============================================================
plt.rcParams.update({
    "font.size": 8,
    "ytick.major.width": 1,
    "xtick.major.width": 1,
    "axes.linewidth": 1,
    "font.family": "Arial",
})

dens_levels = np.sort(ppc_df["density"].unique())
order_levels = np.sort(ppc_df["order"].unique())

nrows = len(dens_levels)
ncols = len(order_levels)

fig_w = max(6.8, 2.2 * ncols)
fig_h = max(3.2, 1.8 * nrows)

# Figure: the matplotlib 'how' lives in locust_density_plots.py.
plots.plot_individual_cell_ppc(ppc_df)


[OK] posterior draws S=4000, animals=440
[OK] mu_re shape=(4000, 440) kappa_re shape=(4000, 440)


Saved figures to: figures/locust/order_density_individual_cell_ppc.pdf; figures/locust/order_density_individual_cell_ppc.png
